# Discount Factor as a Regularizer (ICML 2020) — Notebook unico

Questo quaderno raccoglie **tutto il codice** della cartella `Discount_as_Regularizer-master/` in un unico posto, ordinato per sezioni:

1. Impostazioni e percorsi
2. Utilità (funzioni comuni + MDP/MRP)
3. Esperimenti tabulari (MRP / Policy Evaluation / Control)
4. Grafici
5. Codice TD3/DDPG (MuJoCo)

> Nota: alcune sezioni richiedono librerie specifiche (es. **ray**, **torch**, **gym**, **mujoco-py**). Se non sono installate, quelle celle falliranno: puoi commentarle o installare i requisiti.

---


In [ ]:
# Percorsi: punta alla repo estratta
import os, sys
REPO_ROOT = r"/mnt/data/Discount_as_Regularizer-master"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Opzionale: per scrivere output e figure in una cartella dedicata al notebook
NOTEBOOK_OUT = os.path.join(REPO_ROOT, "saved", "notebook_runs")
os.makedirs(NOTEBOOK_OUT, exist_ok=True)

print("REPO_ROOT =", REPO_ROOT)
print("NOTEBOOK_OUT =", NOTEBOOK_OUT)


## 1) Utilità (comuni, probabilità, MDP/MRP, pianificazione, apprendimento)


In [ ]:
# ===== File: utils/common_utils.py =====

from datetime import datetime
import os
import subprocess
import pathlib
import numpy as np
import random
import sys
import pickle
import shutil
import glob
from pathlib import Path
import json
import matplotlib.pyplot as plt
from copy import deepcopy
import string


# -----------------------------------------------------------------------------------------------------------#
#  Useful functions
# -----------------------------------------------------------------------------------------------------------#


def start_ray(local_mode):
    import ray
    ray.shutdown()
    gettrace = getattr(sys, 'gettrace', None)
    is_debug =  gettrace is not None and gettrace()
    if is_debug and not local_mode:
        Warning('Debugging can not run when in parrnell mode (local_mode==False). \n Setting local_mode=True.')
        local_mode = True
    # end if
    ray.init(local_mode=local_mode, ignore_reinit_error=True)
    if local_mode:
        print('Non-parrnell run (local_mode == True)')
    # end if
# end def
# -----------------------------------------------------------------------------------------------------------#


def save_fig(run_name, base_path='./'):
    ensure_dir(base_path)
    save_path = os.path.join(base_path, run_name)
    plt.savefig(save_path + '.pdf', format='pdf', bbox_inches='tight')
    try:
        plt.savefig(save_path + '.pgf', format='pgf', bbox_inches='tight')
    except:
        print('Failed to save .pgf file  \n  tto allow to save pgf files -  $ sudo apt install texlive-xetex')
    print('Figure saved at ', save_path)
# -----------------------------------------------------------------------------------------------------------#


def set_default_plot_params():
    plt_params = {'font.size': 10,
                  'lines.linewidth': 2, 'legend.fontsize': 16, 'legend.handlelength': 2,
                  'pdf.fonttype': 42, 'ps.fonttype': 42,
                  'axes.labelsize': 18, 'axes.titlesize': 18,
                  'xtick.labelsize': 14, 'ytick.labelsize': 14}
    plt.rcParams.update(plt_params)
# -----------------------------------------------------------------------------------------------------------#


def convert_args(args):
    # convert some of the args from string to a python variable

    fields_to_convert = {
        'critic_hiddens', 'actor_hiddens', 'param_grid_def', 'n_traj_grid', 'mdp_def', 'train_sampling_def',
        'learning_rate_def'}

    for key in args.__dict__:
        val = args.__dict__[key]
        if val is not None:
            if key in fields_to_convert:
                try:
                    val = val.replace("\'", "\"")
                    args.__dict__[key] = json.loads(val)
                except:
                    raise Exception("Failed to read " + str(key))
    return args
# -----------------------------------------------------------------------------------------------------------#


def set_random_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
    except ImportError as e:
        pass  # module doesn't exist
# -----------------------------------------------------------------------------------------------------------#


def get_grid(param_grid_def):
    spacing_type =  param_grid_def['spacing']
    if isinstance(spacing_type, tuple):
        spacing_type = spacing_type[0]
    if spacing_type == 'linspace':
        alg_param_grid = np.linspace(param_grid_def['start'], param_grid_def['stop'],
                                     num=int(param_grid_def['num']))

    elif spacing_type == 'arange':
        alg_param_grid = np.arange(param_grid_def['start'], param_grid_def['stop'])

    elif spacing_type == 'endpoints':
        alg_param_grid = np.linspace(param_grid_def['start'], param_grid_def['end'],
                                    num=int(param_grid_def['num']), endpoint=True)

    elif spacing_type == 'list':
        alg_param_grid = np.asarray(param_grid_def['list'])

    elif spacing_type == 'list_tuples':
        alg_param_grid = param_grid_def['list']

    else:
        raise AssertionError('Invalid param_grid_def')

    if 'decimals' in param_grid_def.keys():
        # truncate decimals:
        alg_param_grid = np.around(alg_param_grid, decimals=param_grid_def['decimals'])

    return alg_param_grid

# ----------------------------------------------------------------------
# -----------------------------------------------------------------------------------------------------------#
# Result saving
# -----------------------------------------------------------------------------------------------------------#


def ensure_dir(dir_path):
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)
# -----------------------------------------------------------------------------------------------------------#


def create_result_dir(args, run_type=None):

    # If run_name empty, set according to time
    time_str = datetime.now().strftime('%Y_%m_%d_%H_%M_%S')
    if args.run_name == '':
        args.run_name = time_str
    run_file_path = sys.argv[0]
    if run_type is None:
        run_file_name = os.path.splitext(os.path.basename(run_file_path))[0]
        run_type = run_file_name
    dir_path = os.path.dirname(os.path.realpath(run_file_path))
    if 'result_dir' not in args:
        args.result_dir = os.path.join(dir_path, 'saved', run_type, args.run_name)
    ensure_dir(args.result_dir)
    message = [
               'Run script: ' + run_file_path,
               'Log file created at ' + time_str,
               'Parameters:', str(args), '-' * 70]
    write_to_log(message, args, mode='w') # create new log file
    write_to_log('Results dir: ' + args.result_dir, args)
    write_to_log('-' * 50, args)
    save_git_status(args.result_dir)
    # save_code(args.result_dir)
    save_run_data(args, {'run_time': 0}, verbose=0)
    pretty_print_args(args)
# ------------------------------------------------------------------------------#


def save_git_status(save_dir):
    # https://towardsdatascience.com/logging-algorithmic-experiments-in-python-9ab1f0e55b89
    # https://stackoverflow.com/a/40895333

    repo_path = str(pathlib.Path(__file__).parent.parent.absolute())

    git_branch = subprocess.check_output(
        ["git", "-C", repo_path, "rev-parse", "--abbrev-ref", "HEAD"]).strip().decode('UTF-8')

    git_commit_short_hash = subprocess.check_output(
        ["git", "-C", repo_path, "describe", "--always"]).strip().decode('UTF-8')

    # diff from last commit:
    git_diff = subprocess.check_output(["git", "-C", repo_path, "diff",  "HEAD^", "HEAD"]).decode('UTF-8')
    # note: you might need to do in project dir $ git reset --hard
    # to correctly track changes from commit

    git_status = subprocess.check_output(["git", "-C", repo_path, "status"]).decode('UTF-8')

    file_path = os.path.join(save_dir, 'git_stat.pkl')
    with open(file_path, 'wb') as f:
        pickle.dump([git_branch, git_commit_short_hash, git_diff, git_status], f)

# ------------------------------------------------------------------------------#


def write_to_log(message, args, update_file=True, mode='a'):
    # mode='a' is append
    # mode = 'w' is write new file
    log_file_path = os.path.join(args.result_dir, 'log') + '.out'
    write_to_file(message, log_file_path, update_file=True, mode='a')

# ------------------------------------------------------------------------------#


def write_to_file(message, log_file_path, update_file=True, mode='a'):
    # mode='a' is append
    # mode = 'w' is write new file
    if not isinstance(message, list):
        message = [message]
    # update log file:
    if update_file:
        with open(log_file_path, mode) as f:
            for string in message:
                print(string, file=f)
    # print to console:
    for string in message:
        print(string)
# -----------------------------------------------------------------------------------------------------------#


def time_now():
    return datetime.now().strftime('%Y\%m\%d, %H:%M:%S')
# -----------------------------------------------------------------------------------------------------------#


def save_run_data(args, info_dict, verbose=1):
    run_data_file_path = os.path.join(args.result_dir, 'run_data.pkl')
    with open(run_data_file_path, 'wb') as f:
        pickle.dump([args, info_dict], f)
    if verbose == 1:
        write_to_log('Results saved in ' + run_data_file_path, args)
# -----------------------------------------------------------------------------------------------------------#


def load_run_data(result_dir, showParams=True):
    run_data_file_path = os.path.join(result_dir, 'run_data.pkl')
    with open(run_data_file_path, 'rb') as f:
       args, info_dict = pickle.load(f)
    print('Data loaded from ', run_data_file_path)
    if showParams:
        print('Parameters \n', args)
    return args, info_dict
# -----------------------------------------------------------------------------------------------------------#


def load_saved_vars(result_dir):
    run_data_file_path = os.path.join(result_dir, 'run_data.pkl')
    with open(run_data_file_path, 'rb') as f:
        loaded_args, loaded_dict = pickle.load(f)
    print('Loaded run parameters: ' + str(loaded_args))
    print('-' * 70)
    return loaded_args, loaded_dict
# -----------------------------------------------------------------------------------------------------------#


def create_results_backup(result_dir):
    src = os.path.join(result_dir, 'run_data.pkl')
    dst = os.path.join(result_dir, 'backup_run_data.pkl')
    shutil.copyfile(src, dst)
    print('Backup of run data with original grid was saved in ', dst)

# -----------------------------------------------------------------------------------------------------------#


def save_code(save_dir):
    # Create backup of code
    source_dir = os.getcwd()
    dest_dir = save_dir + '/Code_Archive/'
    ensure_dir(dest_dir)

    for filename in glob.glob(os.path.join(source_dir, '*.*')):
        if ".egg-info" not in filename and ".py" in filename:
            shutil.copy(filename, dest_dir)

# -----------------------------------------------------------------------------------------------------------#


def get_project_root():
    """Returns project root folder."""
    return str(Path(__file__).parent.parent)
# -----------------------------------------------------------------------------------------------------------#


def bold(s):
    return '\033[1m ' + s + '\033[0m'
# -----------------------------------------------------------------------------------------------------------#


def pretty_print_args(args, printFlag=True):
    from termcolor import colored
    arg_dict = args.__dict__
    arg_keys = list(arg_dict.keys())
    arg_keys.sort(key=lambda k: k.lower())  # sort alphabetically (ignore letter case)
    s = '{'
    for key in arg_keys:
        s += colored(bold('"' + key + '"'), 'blue') + ': ' + str(arg_dict[key]) + ', '
    s += '}'
    if printFlag:
        print(s)
    return s




In [ ]:
# ===== File: utils/prob_utils.py =====
import numpy as np
import scipy.stats as sps

import random
from utils.common_utils import write_to_log

def sample_discrete(probs):
    """
    Samples a discrete distribution
     Parameters:
        probs - probabilities over {0,...K-1}
    Returns:
        drawn random integer from  {0,...K-1} according to probs
    """
    K = probs.size
    return np.random.choice(K, size=1, p=probs)[0]


# ------------------------------------------------------------------------------------------------------------~
def draw_prob_at_dist(n, d=0.):
    """
    Randomly generates a discrete distribution of dimension n with L1 distance of d from the uniform distribution.
    """
    if d == 0.:
        return np.ones(n) / n
    diffs = np.random.uniform(size=n)
    diffs = diffs - diffs.mean() # center to make sum(diffs)==0
    diffs = diffs * d / np.sum(np.abs(diffs))  # scale distance
    probs = diffs + (1/n)  # add diffs to a uniform prob
    probs = simplex_projection(probs)
    assert np.all(0 <= probs)
    return probs

# ------------------------------------------------------------------------------------------------------------~
def generate_discrete_prob_TV(args, n, tv_dist_nrml=0., tol=1e-1, max_iter=5e6):
    """
    Randomly generates a discrete distribution of dimension n with L1 distance of d from the uniform distribution.
    assumption tv_dist is normalized in [0,1]
    """
    assert 0 <= tv_dist_nrml <= 1

    if tv_dist_nrml == 0.:
        return np.ones(n) / n

    tv_dist_max = 0.5 * 2 * (n - 1) / n  # note: tv_dist = 0.5 * L1 dist
    tv_dist = tv_dist_nrml * tv_dist_max  # de-normalize

    done = False
    i = 0
    probs = None
    best_probs = None
    best_err = float('inf')
    l1_dist = 2 * tv_dist # the desired L1 dist
    best_tv_dist_nrml = None
    while not done:
        i += 1
        # probs = sample_simplex(n)   # unifromly
        probs = draw_prob_at_dist(n, l1_dist)
        #  Check that we indeed got the correct L1 distance from a uniform distrib
        curr_l1_dist = np.sum(np.abs(probs-1/n))
        curr_tv_dist_nrml = 0.5 * curr_l1_dist / tv_dist_max
        curr_err = np.abs(curr_tv_dist_nrml - tv_dist_nrml)
        if curr_err < tol:
            done = True
        if curr_err < best_err:
            best_err = curr_err
            best_probs = probs
            best_tv_dist_nrml = curr_tv_dist_nrml
        if i >= max_iter:
            write_to_log(['rejection sampling failed -', 'desired tv_dist [normalized]: ', tv_dist_nrml,
                          ', best_tv_dist_nrml: ', best_tv_dist_nrml, 'best_err: ', best_err], args)
            probs = best_probs
            done = True
        # if i >= max_iter:
        #     write_to_log(['n ', n, 'tv_dist ', tv_dist, 'tol ', tol, 'i ', i,
        #     'desired l1_dist', l1_dist, 'last l1_dist', curr_l1_dist], args)
        #     raise AssertionError('rejection sampling failed')
    # print(i)
    return probs

# ------------------------------------------------------------------------------------------------------------~

# ------------------------------------------------------------------------------------------------------------~
def generate_prob_given_entropy(args, n, ent_nrml, tol=1e-1, max_iter=5e6):
    """
    Randomly generates a discrete distribution of dimension n with given entropy ent.
    assumption ent is normalized in [0,1]
    """
    assert 0. <= ent_nrml <= 1.

    if ent_nrml == 1.:
        return np.ones(n) / n

    ent_max = np.log2(n)
    ent = ent_nrml * ent_max  # de-normalize

    # use rejection sampling
    done = False
    i = 0
    probs = None
    best_probs = None
    best_err = float('inf')
    best_ent_nrml = None
    while not done:
        i += 1
        probs = sample_simplex(n)
        curr_ent = sps.entropy(probs, base=2)
        curr_ent_nrml = curr_ent / ent_max
        cur_err = np.abs(ent_nrml - curr_ent_nrml)
        if cur_err < best_err:
            best_err = cur_err
            best_probs = probs
            best_ent_nrml= curr_ent_nrml
        if cur_err < tol:
            done = True
        if i >= max_iter:
            write_to_log(['rejection sampling failed -', 'desired ent: ', ent, 'best_err: ', best_err, ', best_ent_nrml: ', best_ent_nrml], args)
            probs = best_probs
            done = True
            # write_to_log(['n ', n, 'ent ', ent, 'tol ', tol, 'i ', i, 'last ent', curr_ent], args)
            # raise AssertionError('rejection sampling failed')
    return probs
# ------------------------------------------------------------------------------------------------------------~

def simplex_projection(s):
    """Projection onto the unit simplex.
    source: https://ee227c.github.io/code/lecture5.html"""
    if np.sum(s) <=1 and np.alltrue(s >= 0):
        return s
    # Code taken from https://gist.github.com/daien/1272551
    # get the array of cumulative sums of a sorted (decreasing) copy of v
    u = np.sort(s)[::-1]
    cssv = np.cumsum(u)
    # get the number of > 0 components of the optimal solution
    rho = np.nonzero(u * np.arange(1, len(u)+1) > (cssv - 1))[0][-1]
    # compute the Lagrange multiplier associated to the simplex constraint
    theta = (cssv[rho] - 1) / (rho + 1.0)
    # compute the projection by thresholding v using theta
    probs = np.maximum(s-theta, 0)
    probs /= probs.sum()  #  correct numerical errors
    return probs

# ------------------------------------------------------------------------------------------------------------~
def sample_simplex(n):
    """
    Randomly (uniform) draw vector from n-simplex
    """
    v = np.random.uniform(size=n)
    v = -np.log(v)
    v /= np.sum(v)
    return v
# ------------------------------------------------------------------------------------------------------------~

# def sample_simplex(n):
#     """
#     Randomly (uniform) draw vector from n-simplex
#     use Kraemer Algorithm
#     https://www.cs.cmu.edu/~nasmith/papers/smith+tromble.tr04.pdf
#     """
#     M = 1000 # the numbers are up to  precision 1/M
#     x = np.random.randint(low=0, high=M+1, size=n-1)
#     x[0] = 0
#     x[-1] = M
#     x.sort()
#     y = x[1:] - x[:-1]
#     probs = y/M
#     return probs




def augment_mix_time(Pss, forced_mix_time):
    nS = Pss.shape[0]
    # Modify P so we have the desired mixing time
    evals, evecs = np.linalg.eig(Pss)
    idx = evals.argsort()[::-1]
    evals = evals[idx]
    evecs = evecs[:, idx]

    lambda2_old = evals[1]
    lambda2_old_abs = np.abs(lambda2_old)
    spect_gap_old = 1 - lambda2_old_abs
    mix_time_old = 1 / spect_gap_old
    mix_time = forced_mix_time
    forced_lamb2_abs = (mix_time - 1.) / mix_time
    forced_spect_gap = 1 - forced_lamb2_abs
    evals[1] = lambda2_old * forced_lamb2_abs / lambda2_old_abs  # update lambda_2
    # for any e.val (besides the first) with abs larger then lamb2_abs - fix it
    for iev in range(1, nS):
        eval_abs = np.abs(evals[iev])
        if eval_abs > forced_lamb2_abs:
            evals[iev] *= forced_lamb2_abs / eval_abs

    # reconstruct P:
    PssNew = np.real(evecs @ np.diag(evals) @ np.linalg.inv(evecs))
    # correct numerical errors:
    PssNew[PssNew < 1e-15] = 0
    row_sums = PssNew.sum(axis=1)
    PssNew = PssNew / row_sums[:, np.newaxis]
    return PssNew


In [ ]:
# ===== File: utils/mdp_utils.py =====
import numpy as np
from utils.prob_utils import sample_discrete, generate_prob_given_entropy, generate_discrete_prob_TV
from utils.planing_utils import get_stationary_distrb, GetUniformPolicy, GetPolicyDynamics
from utils.prob_utils import augment_mix_time

# Markov Decision Process
# ------------------------------------------------------------------------------------------------------------~


def SetMdpArgs(args):

	if 'mdp_def' in args:
		mdp_def = args.mdp_def
	else:
		mdp_def = args.mrp_def
	mdp_type = mdp_def['type']
	args.mdp_def = mdp_def
	if mdp_type == 'RandomMDP':
		args.nS = mdp_def['S']  # number of states
		args.nA = mdp_def['A']  # number of actions
		args.k = mdp_def['k']  # Number of non-zero entries in each row  of transition-matrix
		args.reward_std = mdp_def['reward_std']
	elif mdp_type == 'GridWorld':
		args.nS = mdp_def['N0'] * mdp_def['N1']
		args.nA = 5
		args.reward_std = mdp_def['reward_std']
	elif mdp_type == 'GridWorld2':
		args.nS = mdp_def['N0'] * mdp_def['N1']
		args.nA = 4
		args.reward_std = mdp_def['reward_std']
	else:
		raise AssertionError('Invalid mdp_type')


#------------------------------------------------------------------------------------------------------------~


class MDP(): # Markov Desicion Process
	def __init__(self, args, init_sampling=True):
		"""
		  Randomly generate an MDP


		  Parameters:

		  Returns:
		  P: [nS x nA x nS] transitions probabilities matrix  P_{s,a,s'}=P(s'|s,a)
		  R: [nS x nA] mean rewards matrix R
		  """
		nS = args.nS  # number of states
		nA = args.nA  # number of actions
		self.nA = nA
		self.nS = nS

		####### Create the MDP
		mdp_type = args.mdp_def['type']
		if mdp_type == 'RandomMDP':
			P, R = self.generate_RandomMDP(args)
		elif mdp_type == 'GridWorld':
			P, R = self.generate_GridWorld(args)
		else:
			raise AssertionError('Invalid mdp_type')

		# Save MDP parameters
		assert np.max(R) <= 1   # we assume Rmax <= 1
		self.R = R
		self.reward_std = args.reward_std
		self.P = P
		self.type = args.mdp_def['type']
		self.define_initial_distrb(args)
		if init_sampling:
			self.define_sampling_method(args)
	# end function
	# ------------------------------------------------------------------------------------------------------------~

	def define_initial_distrb(self, args):
		nS = self.nS
		nA = self.nA
		if 'initial_state_distrb' in args.mdp_def.keys():
			initial_state_distrb = args.mdp_def['initial_state_distrb']
			if initial_state_distrb == 'uniform':
				self.initial_state_distrb = np.ones(nS) / nS  # uniform
			elif initial_state_distrb == 'state_0':
				self.initial_state_distrb = np.zeros(nS)
				self.initial_state_distrb[0] = 1.
			else:
				raise AssertionError('Invalid initial_state_distrb')
		else:
			self.initial_state_distrb = np.ones(nS) / nS  # uniform
	# ------------------------------------------------------------------------------------------------------------~

	def define_sampling_method(self, args):
		sampling_def = args.train_sampling_def
		nS = self.nS
		nA = self.nA

		if sampling_def['type'] == 'Trajectories':
			pass  # in this case we will later sample a sequence trajectories according to some given policy

		elif sampling_def['type'] == 'Generative_Stationary':
			pass  # in this case we will later sample a iid draws from the stationary distribution according to some given policy

		elif sampling_def['type'] == 'sample_all_s_a':
			# in this case we will later sample all (s,a) X number of trajectories
			pass

		elif sampling_def['type'] == 'chain_mix_time':
			# in this case the data generation is done by a chain we set to have some mix time (the policy is ignored)
			pi_unif = GetUniformPolicy(nS, nA)
			Pss, Rs = GetPolicyDynamics(self.P, self.R, pi_unif)
			# Modify P so we have the desired mixing time
			mix_time = args.train_sampling_def['mix_time']
			Pss = augment_mix_time(Pss, mix_time)
			self.Pss_fixed = Pss

		elif sampling_def['type'] == 'Generative':
			# Create a fixed sampling distribution
			# Create sampling distribution for states
			if 'states_TV_dist_from_uniform' in sampling_def.keys():
				tv_dist = sampling_def['states_TV_dist_from_uniform']
				probs_s = generate_discrete_prob_TV(args, nS, tv_dist_nrml=tv_dist)
			elif 'states_entropy' in sampling_def.keys():
				ent_S = sampling_def['states_entropy']
				probs_s = generate_prob_given_entropy(args, nS, ent_S)
			else:
				raise AssertionError('Unrecognized sampling_def')
			# end if
			# Create sampling distribution for actions
			if 'actions_TV_dist_from_uniform' in sampling_def.keys():
				tv_dist = sampling_def['actions_TV_dist_from_uniform']
				probs_a = generate_discrete_prob_TV(args, nA, tv_dist_nrml=tv_dist)
			elif 'actions_entropy' in sampling_def.keys():
				ent_A = sampling_def['actions_entropy']
				probs_a = generate_prob_given_entropy(args, nA, ent_A)
			else:
				raise AssertionError('Unrecognized sampling_def')
			self.probs_s = probs_s
			self.probs_a = probs_a

		elif sampling_def['type'] == 'Generative_uniform':
			# Create fixed uniform sampling distribution
			probs_s = generate_discrete_prob_TV(args, nS, tv_dist_nrml=0)
			probs_a = generate_discrete_prob_TV(args, nA, tv_dist_nrml=0)
			self.probs_s = probs_s
			self.probs_a = probs_a

		else:
			raise AssertionError('Unrecognized sampling_def')
		# end if
		self.sampling_type =  sampling_def['type']
	# end def
	# ------------------------------------------------------------------------------------------------------------~


	def SampleData(self, args, pi, n_traj, p0=None):
		"""
		# generate n trajectories

		Parameters:
		P: [nS x nA x nS] transitions probabilities matrix  P_{s,a,s'}=P(s'|s,a)
		R: [nS x nA] mean rewards matrix R
		pi: [nS x nA]  matrix representing  pi(a|s)
		n: number of trajectories to generate
		depth: Length of trajectory
		p0 (optional) [nS] matrix of initial state distribution
		Returns:
		data: list of n trajectories, each is a list of sequence of depth tuples (state, action, reward, next state)
		"""
		R = self.R
		P = self.P
		nS = self.nS
		nA = self.nA
		reward_std = self.reward_std
		depth = args.depth
		if p0 is None:
			p0 = self.initial_state_distrb
		data = []
		sampling_type = self.sampling_type

		if sampling_type == 'Trajectories':
			# generate sampled trajectories
			for i_traj in range(n_traj):
				data.append([])
				# sample initial state:
				s = sample_discrete(p0)
				a = sample_discrete(pi[s, :])
				# Until t==depth, sample a~pi(.|s), s'~P(.|s,a), r~R(s,a)
				for t in range(depth):
					if self.is_transitions_deterministic == 'True':
						# deterministic transition
						s_next = self.next_state_table[s, a]
					else:
						s_next = sample_discrete(P[s, a, :])
					a_next = sample_discrete(pi[s_next, :])
					r = R[s, a] + np.random.randn(1)[0] * reward_std
					data[i_traj].append((s, a, r, s_next, a_next))
					s = s_next
					a = a_next
				# end for t
			# end for i_traj

		elif sampling_type in {'Generative', 'Generative_uniform'}:
			#  iid sampling according to distribution that was set in the init of current MDP
			probs_s = self.probs_s
			probs_a = self.probs_a
			# Sample data
			for i_traj in range(n_traj):
				data.append([])
				for t in range(depth):
					s = sample_discrete(probs_s)
					a = sample_discrete(probs_a)
					r = R[s, a] + np.random.randn(1)[0] * reward_std
					s_next = sample_discrete(P[s, a, :])
					a_next = sample_discrete(pi[s_next, :])
					# s_next and  a_next is not used in the new iteration since we generate iid samples
					data[i_traj].append((s, a, r, s_next, a_next))
				# end for t
			# end for i_traj

		elif sampling_type == 'Generative_Stationary':
			# iid sampling distribution accordign to the stationary distribution of pi
			probs_s = get_stationary_distrb(self, pi)
			# Sample data
			for i_traj in range(n_traj):
				data.append([])
				for t in range(depth):
					s = sample_discrete(probs_s)
					probs_a = pi[s]
					a = sample_discrete(probs_a)
					r = R[s, a] + np.random.randn(1)[0] * reward_std
					s_next = sample_discrete(P[s, a, :])
					a_next = sample_discrete(pi[s_next, :])
					data[i_traj].append((s, a, r, s_next, a_next))
				# end for t
			# end for i_traj

		elif sampling_type == 'sample_all_s_a':
			for i_traj in range(n_traj):
				data.append([])
				for s in range(nS):
					for a in range(nA):
						r = R[s, a] + np.random.randn(1)[0] * reward_std
						s_next = sample_discrete(P[s, a, :])
						a_next = sample_discrete(pi[s_next, :])
						data[i_traj].append((s, a, r, s_next, a_next))
					# end for a
				# end for s
			# end for i_traj

		elif sampling_type == 'chain_mix_time':
			Pss = self.Pss_fixed
			# generate sampled trajectories
			for i_traj in range(n_traj):
				data.append([])
				# sample initial state:
				s = sample_discrete(p0)
				a = sample_discrete(pi[s, :])
				# Until t==depth, sample a~pi(.|s), s'~Pss(.|s), r~R(s,a)
				# policy is ignored in the dynamics
				for t in range(depth):
					s_next = sample_discrete(Pss[s, :])
					a_next = sample_discrete(pi[s_next, :])
					r = R[s, a] + np.random.randn(1)[0] * reward_std
					data[i_traj].append((s, a, r, s_next, a_next))
					s = s_next
					a = a_next
				# end for t
			# end for i_traj
		else:
			raise AssertionError('Unrecognized data_type')
		return data
# ------------------------------------------------------------------------------------------------------------~


	def generate_RandomMDP(self, args):

		nS = self.nS
		nA = self.nA
		P = np.zeros((nS, nA, nS))
		# For each state-action pair (s; a), the distribution over the next state,  P_{s,a,s'}=P(s'|s,a), is determined by choosing k  non-zero entries uniformly from
		#  all nS states, filling these k entries with values uniformly drawn from [0; 1], and finally normalizing
		k = args.k  # Number of non-zero entries in each row  of transition-matrix
		self.is_transitions_deterministic = False
		for a in range(nA):
			for i in range(nS):
				nonzero_idx = np.random.choice(nS, k, replace=False)
				for j in nonzero_idx:
					P[i, a, j] = np.random.rand(1)
				P[i, a, :] /= P[i, a, :].sum()
		R = np.random.rand(nS, nA)  # rewards means
		return P, R
# ------------------------------------------------------------------------------------------------------------~
	def generate_GridWorld(self, args):

		nS = self.nS
		nA = self.nA
		forward_prob_distrb = args.mdp_def['forward_prob_distrb']
		self.is_transitions_deterministic = forward_prob_distrb == 1
		N0 = args.mdp_def['N0']
		N1 = args.mdp_def['N1']
		action_set = [[-1, 0], [1, 0], [0, 1], [0, -1], [0, 0]]
		assert args.nA == len(action_set)
		P = np.zeros((nS, nA, nS))
		next_state_table = np.empty((nS, nA), dtype=np.int)

		##### Create reward means: ####
		R_image = (args.mdp_def['R_high'] - args.mdp_def['R_low']) * np.random.rand(nS) + args.mdp_def['R_low']  # draw reward means
		# add goal state
		if args.mdp_def['goal_reward'] is not None:
			goal_reward = args.mdp_def['goal_reward']
			s_goal = np.random.randint(nS)
			R_image[s_goal] = goal_reward
		# translate to R(s,a) matrix
		R = np.zeros((nS, nA))
		for s in range(nS):
			# set same reward mean for all actions in each state
			R[s, :] = R_image[s]

		##### set state transition probs #####
		for s0 in range(N0):
			for s1 in range(N1):
				s = s0 + s1 * N0
				for a, shift in enumerate(action_set):
					s_next0 = s0 + shift[0]
					s_next1 = s1 + shift[1]
					if 0 <= s_next0 < N0 and 0 <= s_next1 < N1 and np.any(action_set[a] != [0, 0]):
						# in case the move is legal and we may change state
						s_next = s_next0 + s_next1 * N0

						if forward_prob_distrb == 1:  # deterministic
							p_forward = 1.
						elif isinstance(forward_prob_distrb, float):  # deterministic
							p_forward = forward_prob_distrb
						elif forward_prob_distrb == 'uniform':  # uniform distribution
							p_forward = np.random.rand(1)
						elif 'beta' in forward_prob_distrb:  # beta distribution
							p_forward = np.random.beta(forward_prob_distrb['alpha'], forward_prob_distrb['beta'],
							                           size=1)
						else:
							raise AssertionError("Invalid args.mdp_def['forward_prob_distrb']")
						# end if

						P[s, a, s_next] = p_forward  # move state probability
						P[s, a, s] = (1. - p_forward)  # stay in place probability
					else:
						# otherwise - stay in place
						s_next = s
						P[s, a, s_next] = 1.
					# end if
					if self.is_transitions_deterministic:
						next_state_table[s, a] = s_next
					# end if
				# end for a
			# end for s1
			# assert np.abs(P[s, a].sum() - 1) < 1e-6
		# end for s0
		if self.is_transitions_deterministic:
			self.next_state_table = next_state_table
		return P, R

In [ ]:
# ===== File: utils/mrp_utils.py =====
import numpy as np
from utils.prob_utils import sample_discrete
from utils.mdp_utils import SetMdpArgs, MDP
from utils.planing_utils import GetUniformPolicy, GetPolicyDynamics
from utils.prob_utils import augment_mix_time, generate_discrete_prob_TV, generate_prob_given_entropy

# Markov Reward Process

# ------------------------------------------------------------------------------------------------------------~


def SetMrpArgs(args):
    mrp_type = args.mrp_def['type']
    if mrp_type == 'ToyMRP':
        args.nS = 2
        args.reward_std = args.mrp_def['reward_std']
    elif mrp_type == 'Chain':
        args.nS = args.mrp_def['length']
        args.reward_std = args.mrp_def['reward_std']
    elif mrp_type in {'RandomMDP', 'GridWorld'}:
        SetMdpArgs(args)
    else:
        raise AssertionError('Invalid mrp_type')
# ------------------------------------------------------------------------------------------------------------~

# ------------------------------------------------------------------------------------------------------------~


class MRP(): # Markov Reward Process
    def __init__(self, args):
        """
          Parameters:

          Returns:

          """
        nS = args.nS  # number of states
        P = np.zeros((nS, nS))
        mrp_type = args.mrp_def['type']

        if mrp_type == 'ToyMRP':
            p01 = args.mrp_def['p01']
            p00 = 1 - p01
            p10 = args.mrp_def['p10']
            p11 = 1 - p10

            P[0, 0] = p00
            P[0, 1] = p01
            P[1, 0] = p10
            P[1, 1] = p11

            R = np.random.rand(nS)

        elif mrp_type == 'Chain':
            p_left = args.mrp_def['p_left']
            p_right = 1 - p_left
            nS = args.mrp_def['length']
            P = np.zeros((nS, nS))
            for i in range(nS):
                if i < nS - 1:
                    P[i, i+1] = p_right
                else:
                    P[i,i] = p_right
                if i > 0:
                    P[i, i - 1] = p_left
                else:
                    P[i, i] = p_left

            mean_reward_range =  args.mrp_def['mean_reward_range']
            R = mean_reward_range[0] + (mean_reward_range[1] - mean_reward_range[0]) * np.random.rand(nS)
            # R = np.random.rand(nS)

        elif mrp_type in {'RandomMDP', 'GridWorld'}:
            mdp = MDP(args, init_sampling=False)
            nS = mdp.nS
            nA = mdp.nA
            if args.mrp_def['policy'] == 'uniform':
                pi = GetUniformPolicy(nS, nA)
            else:
                raise AssertionError('Unrecognized policy def')
            P, R = GetPolicyDynamics(mdp.P, mdp.R, pi)

            # Modify P so we have the desired mixing time
            if 'forced_mix_time' in args:
                P = augment_mix_time(P, args.forced_mix_time)


        else:
            raise AssertionError('Invalid mrp_type')

        self.R = R
        self.P = P
        self.nS = nS
        self.type = args.mrp_def['type']
        self.define_sampling_method(args)
        self.reward_std = args.reward_std
        if args.initial_state_distrb_type == 'middle':
            self.initial_state_distrb = np.zeros(nS)
            self.initial_state_distrb[nS // 2] = 1.
        elif args.initial_state_distrb_type == 'uniform':
            self.initial_state_distrb = np.ones(nS) / nS
        else:
            raise AssertionError("Unrecognized initial_state_distrb_type")

    # ------------------------------------------------------------------------------------------------------------~


    def define_sampling_method(self, args):
        sampling_def = args.train_sampling_def
        nS = self.nS
        sampling_type = sampling_def['type']
        if sampling_type == 'Trajectories':
            pass  # in this case we will later sample a sequence trajectories according to some given policy
        elif sampling_type == 'sample_all_s':
            # in this case we will later sample all (s) X number of trajectories
            pass
        elif sampling_type == 'Generative':
            # Create a fixed sampling distribution
            # Create sampling distribution for states
            if 'states_TV_dist_from_uniform' in sampling_def.keys():
                tv_dist = sampling_def['states_TV_dist_from_uniform']
                probs_s = generate_discrete_prob_TV(args, nS, tv_dist_nrml=tv_dist)
            elif 'states_entropy' in sampling_def.keys():
                ent_S = sampling_def['states_entropy']
                probs_s = generate_prob_given_entropy(args, nS, ent_S)
            else:
                raise AssertionError('Unrecognized sampling_def')
            # end if
            self.probs_s = probs_s

        elif sampling_type == 'Generative_uniform':
            # Create fixed uniform sampling distribution
            probs_s = generate_discrete_prob_TV(args, nS, tv_dist_nrml=0)
            self.probs_s = probs_s

        else:
            raise AssertionError('Unrecognized sampling_def')
        # end if
        self.sampling_type = sampling_type
    # end def
    # ------------------------------------------------------------------------------------------------------------~


    def SampleDataMrp(self, args, p0=None):
        """
        # generate n trajectories

        Parameters:
        """
        n_traj = args.n_trajectories
        R = self.R
        P = self.P
        nS = self.nS
        reward_std = self.reward_std
        sampling_type = self.sampling_type
        depth = args.depth  # trajectory length
        if p0 is None:
            p0 = self.initial_state_distrb
        data = []
        if sampling_type == 'Trajectories':
            for i_traj in range(n_traj):
                data.append([])
                # sample initial state:
                s = sample_discrete(p0)
                for t in range(depth):
                    # Until t==depth, sample a~pi(.|s), s'~P(.|s,a), r~R(s,a)
                    s_next = sample_discrete(P[s, :])
                    r = R[s] + np.random.randn(1)[0] * reward_std
                    data[i_traj].append((s, r, s_next))
                    s = s_next
                # end for t
            # end for i_traj

        elif sampling_type in {'Generative', 'Generative_uniform'}:
            #  iid sampling according to distribution that was set in the init of current MDP
            probs_s = self.probs_s
            # Sample data
            for i_traj in range(n_traj):
                data.append([])
                for t in range(depth):
                    s = sample_discrete(probs_s)
                    r = R[s] + np.random.randn(1)[0] * reward_std
                    s_next = sample_discrete(P[s, :])
                    data[i_traj].append((s, r, s_next))
            # end for t
        # end for i_traj
        elif sampling_type == 'sample_all_s':
            for i_traj in range(n_traj):
                data.append([])
                for s in range(nS):
                    r = R[s] + np.random.randn(1)[0] * reward_std
                    s_next = sample_discrete(P[s, :])
                    data[i_traj].append((s, r, s_next))
                # end for s
            # end for i_traj
        else:
            raise AssertionError('Unrecognized data_type')
        return data


# ------------------------------------------------------------------------------------------------------------~


def calc_typical_mixing_time(P):
    evals, evecs = np.linalg.eig(P)
    evals_abs = np.abs(evals)
    evals_abs.sort()
    mixing_time = 1 / (1-evals_abs[-2]) # inverse spectral gap , note: evals_abs[-1] always == 1
    return mixing_time
# ------------------------------------------------------------------------------------------------------------~



In [ ]:
# ===== File: utils/planing_utils.py =====
import numpy as np
from scipy import stats
from utils.prob_utils import sample_simplex

#------------------------------------------------------------------------------------------------------------~


def generalized_argmax_indicator(x):
    argmax_size = np.sum(x == x.max())
    indc = np.zeros_like(x)
    indc[x == x.max()] = 1 / argmax_size
    return indc


#------------------------------------------------------------------------------------------------------------~


def generalized_greedy(Q):
    """
    Calculates a greedy policy w.r.t Q
    if all Q values are distinct then we derive a deterministic policy.
    If several Q values are equal, the probability is divided among them

    Parameters:
     Q [S x A]  Q-function

    Returns:
    pi: [S x A]  matrix representing  pi(a|s)
    """
    if Q.ndim != 2:
        raise AssertionError('Invalid input')
    S = Q.shape[0]
    A = Q.shape[1]
    pi = np.zeros((S,A))
    for s in range(S):
        pi[s] = generalized_argmax_indicator(Q[s])
    return pi

#------------------------------------------------------------------------------------------------------------~


def GetUniformPolicy(nS, nA):
    """
    Create a Markov stochastic policy which chooses actions randomly uniform from each state

    Parameters:
    nS: number of states
    nA: number of actions

    Returns:
    pi: [S x A]  matrix representing  pi(a|s)
    """
    pi = np.ones((nS, nA))
    for i in range(nS):
        pi[i] /= pi[i].sum()
    return pi
#-------------------------------------------------------------------------------------------


def draw_policy_at_random(nS, nA):
    pi = np.zeros((nS, nA))
    for i in range(nS):
        pi[i] = sample_simplex(nA)
    return pi
#------------------------------------------------------------------------------------------------------------~


def GetPolicyDynamics(P, R, pi):
    """
    Calculate the dynamics when following the policy pi

    Parameters:
    P: [nS x nA x nS] transitions probabilities matrix  P_{s,a,s'}=P(s'|s,a)
    R: [nS x nA] mean rewards matrix R
    pi: [nS x nA]  matrix representing  pi(a|s)

    Returns:
    P_pi: [nS x nS] transitions matrix  when following the policy pi      (P_pi)_{s,s'} P^pi(s'|s)
    R_pi: [nS] mean rewards at each state when following the policy pi    (R_pi)_{s} = R^pi(s)
    """
    if P.ndim != 3 or R.ndim != 2 or pi.ndim != 2:
        raise AssertionError('Invalid input')
    nS = P.shape[0]
    nA = P.shape[1]
    P_pi = np.zeros((nS, nS))
    R_pi = np.zeros((nS))
    for i in range(nS):  # current state
        for a in range(nA):
            for j in range(nS): # next state
                # Explanation: P(s'|s) = sum_a pi(a|s)P(s'|s,a)
                P_pi[i, j] += pi[i, a] * P[i,a,j]
                is_row_updated = True
            R_pi[i] += pi[i, a] * R[i,a]

    if np.any(np.abs(P_pi.sum(axis=1) - 1) > 1e-5):
        raise RuntimeError('Probabilty matrix not normalized!!')
    return P_pi, R_pi

#------------------------------------------------------------------------------------------------------------~


def PolicyEvaluation(M, pi, gamma, P_pi=None, R_pi=None):
    """
    Calculates the value-function for a given policy pi and a known model

    Parameters:
    P: [nS x nA x nS] transitions probabilities matrix  P_{s,a,s'}=P(s'|s,a)
    R: [nS x nA] mean rewards matrix R
    pi: [nS x nA]  matrix representing  pi(a|s)
    gamma: Discount factor

    Returns:
    V_pi: [nS] The value-function for a fixed policy pi, i,e. the the expected discounted return when following pi starting from some state
    Q_pi [nS x nA] The Q-function for a fixed policy pi, i,e. the the expected discounted return when following pi starting from some state and action
    """
    # (1) Use PolicyDynamics to get P and R, (2) V = (I-gamma*P)^-1 * R
    P = M.P
    R = M.R
    if P.ndim != 3 or R.ndim != 2 or pi.ndim != 2:
        raise AssertionError('Invalid input')
    nS = P.shape[0]
    nA = P.shape[1]
    if P_pi is None or R_pi is None:
        P_pi, R_pi = GetPolicyDynamics(P, R, pi)
    V_pi = np.linalg.solve((np.eye(nS) - gamma * P_pi), R_pi)
    # Verify that R_pi + gamma * np.matmul(P_pi,  V_pi) == V_pi
    Q_pi = np.zeros((nS, nA))
    for a in range(nA):
        for i in range(nS):
            Q_pi[i, a] = R[i, a] + gamma * np.matmul(P[i,a,:], V_pi)
    # Verify that V_pi(s) = sum_a pi(a|s) * Q_pi(s,a)
    return V_pi, Q_pi
#------------------------------------------------------------------------------------------------------------~


def PolicyIteration(M, gamma):
    """
       Finds the optimal policy given a known model using policy-iteration algorithm

       Parameters:
       P: [nS x nA x nS] transitions probabilities matrix  P_{s,a,s'}=P(s'|s,a)
       R: [nS x nA] mean rewards matrix R
       gamma: Discount factor

       Returns
       pi_opt [nS x nA]: An optimal policy (assuming given model and gamma)
       V_opt: [nS] The optimal value-function , i,e. the the expected discounted return when following optimal policy  starting from some state
       Q_opt [nS x nA] The optimal Q-function, i,e. the the expected discounted return when following optimal policy starting from some state and action
       """

    # The algorithm: until policy not changes: (1) run policy-evaluation to get Q_pi  (2) new_policy = argmax Q
    nS = M.nS
    nA = M.nA
    Q_pi = np.zeros((nS, nA))
    # initial point of the algorithm: uniform policy
    pi = np.ones((nS, nA)) / nA
    pi_prev = nA - pi # arbitrary different policy than pi
    max_iter = nS*nA
    iter = 0
    while np.any(pi != pi_prev):
        pi_prev = pi
        _, Q_pi = PolicyEvaluation(M, pi, gamma)
        # Policy improvement:
        # pi = np.zeros((nS, nA))
        # pi[np.arange(nS), np.argmax(Q_pi, axis=1)] = 1  #  set 1 for the optimal action w.r.t Q, and 0 for the other actions
        pi = generalized_greedy(Q_pi)
        if iter > max_iter:
            raise RuntimeError('Policy Iteration should have stopped by now!')
        iter += 1

    pi_opt = pi
    Q_opt = Q_pi
    V_opt = np.max(Q_opt, axis=1)
    return pi_opt,  V_opt, Q_opt
# ------------------------------------------------------------------------------------------------------------~


def PolicyIteration_GivenRP(R, P, gamma, args):
    from utils.mdp_utils import MDP
    M = MDP(args)
    M.R = R
    M.P = P
    return PolicyIteration(M, gamma)
# ------------------------------------------------------------------------------------------------------------~


def ValueOfGreedyPolicyForV(V, M, gammaEval):
    P = M.P
    R = M.R
    nS = M.nS
    nA = M.nA
    Q_pi = np.zeros((nS, nA))
    for a in range(nA):
        for i in range(nS):
            Q_pi[i, a] = R[i, a] + gammaEval * np.matmul(P[i, a, :], V)
    # pi = np.zeros((nS, nA))
    # pi[np.arange(nS), np.argmax(Q_pi, axis=1)] = 1  # set 1 for the optimal action w.r.t Q, and 0 for the other actions
    pi = generalized_greedy(Q_pi)
    Vgreedy, _ = PolicyEvaluation(M, pi, gammaEval)
    return Vgreedy
# ------------------------------------------------------------------------------------------------------------~


def BellmanErr(V, M, pi, gammaEval):
    P = M.P
    R = M.R
    P_pi, R_pi = GetPolicyDynamics(P, R, pi)
    errVec = V - (R_pi + gammaEval * np.matmul(P_pi, V))
    return errVec
# ------------------------------------------------------------------------------------------------------------~


def evaluate_value_estimation(loss_type, V_pi, V_est, M, pi, gammaEval, gamma_guidance):
    if loss_type == 'correction_scaling':
        # Correction factor:
        V_est = V_est * (1 / (1 - gammaEval)) / (1 / (1 - gamma_guidance))
        eval_loss = np.abs(V_pi - V_est).mean()

    elif loss_type =='L1_normalized':
        V_est_norm = np.sum(np.abs(V_est))
        V_pi_norm = np.sum(np.abs(V_pi))
        eval_loss = np.abs(V_pi / V_pi_norm - V_est / V_est_norm).mean()

    # elif loss_type =='Lmax_normalized':
    #     V_est_norm = np.max(np.abs(V_est))
    #     V_pi_norm = np.max(np.abs(V_pi))
    #     eval_loss = np.abs(V_pi / V_pi_norm - V_est / V_est_norm).mean()

    elif loss_type =='Lmax':
        eval_loss = np.abs(V_pi - V_est).max()


    elif loss_type =='L2_normalized':
        V_est_norm = np.linalg.norm(V_est)
        V_pi_norm = np.linalg.norm(V_pi)
        eval_loss = np.sqrt(np.square(V_pi / V_pi_norm - V_est / V_est_norm).mean())

    elif loss_type =='L2':
        eval_loss = np.sqrt(np.square(V_pi - V_est).sum())

    elif loss_type =='L2_uni_weight':
        eval_loss = np.sqrt(np.mean(np.square(V_pi - V_est)))


    elif loss_type == 'one_pol_iter_l2_loss':
        # Optimal policy for the MDP:
        pi_opt, V_opt, Q_opt = PolicyIteration(M, gammaEval)
        V_est_g = ValueOfGreedyPolicyForV(V_est, M, gammaEval)
        eval_loss = (np.square(V_opt - V_est_g)).mean()

    elif loss_type == 'greedy_V_L1':
        V_pi_g = ValueOfGreedyPolicyForV(V_pi, M, gammaEval)
        V_est_g = ValueOfGreedyPolicyForV(V_est, M, gammaEval)
        eval_loss = np.abs(V_pi_g - V_est_g).mean()

    elif loss_type == 'greedy_V_L_infty':
        V_pi_g = ValueOfGreedyPolicyForV(V_pi, M, gammaEval)
        V_est_g = ValueOfGreedyPolicyForV(V_est, M, gammaEval)
        eval_loss = np.abs(V_pi_g - V_est_g).max()

    elif loss_type =='Bellman_Err':
        # V_pi_err = BellmanErr(V_pi, P, R, pi, gammaEval) # should be very close to 0
        V_est_err = BellmanErr(V_est, M, pi, gammaEval)
        eval_loss = np.abs(V_est_err).mean()

    elif loss_type == 'Values_SameGamma':
        V_pi_gamma, _ = PolicyEvaluation(M, pi, gamma_guidance)
        eval_loss = np.abs(V_est - V_pi_gamma).mean()

    elif loss_type == 'rankings_kendalltau':
        tau, p_value = stats.kendalltau(V_est, V_pi)
        eval_loss = -tau
    elif loss_type == 'rankings_spearmanr':
        rho, pval = stats.spearmanr(V_est, V_pi)
        eval_loss = -rho
    # elif loss_type == 'greedy_V_L_infty_MRP':
    #     V_pi_g = ValueOfGreedyPolicyForV(V_pi, M, gammaEval)
    #     V_est_g = ValueOfGreedyPolicyForV(V_est, M, gammaEval)
    #     eval_loss = np.abs(V_pi_g - V_est_g).max()

    else:
        raise AssertionError('unrecognized evaluation_loss_type')
    return eval_loss


# ------------------------------------------------------------------------------------------------------------~
def get_stationary_distrb(M, pi):
    P_pi, R_pi = GetPolicyDynamics(M.P, M.R, pi)
    evals, evecs = np.linalg.eig(P_pi.T)  # det left eigenvectors of P_pi
    evals_abs = np.abs(evals)
    sort_inds = np.argsort(evals_abs)
    ind_of_largest = sort_inds[-1]
    dVec = evecs[:, ind_of_largest].copy()
    dVec = np.real_if_close(dVec)
    dVec /= dVec.sum()

    ### DEBUG : verify that p -> dVec ######
    # p = np.ones(M.nS) / M.nS
    # for i in range(1000):
    #     p = p @ P_pi
    # print(p)
    ######
    return dVec


In [ ]:
# ===== File: utils/learning_utils.py =====
import numpy as np
from numpy import matlib
from random import randrange
from utils.planing_utils import GetUniformPolicy, generalized_greedy, PolicyIteration_GivenRP, draw_policy_at_random


# ------------------------------------------------------------------------------------------------------------~
def run_learning_method(args_r, M, n_traj, gamma_guidance, l2_factor, l1_factor):
    if args_r.method in {'Expected_SARSA','SARSA', 'LSTDQ', 'ELSTDQ', 'LSTDQ_nested', 'ELSTDQ_nested'}:
        pi_t = model_free_approx_policy_iteration(args_r, M, n_traj, gamma_guidance, l2_factor, l1_factor)

    elif args_r.method == 'Model_Based':
        pi_t = model_based_learning(args_r, M, n_traj, gamma_guidance, l2_factor, l1_factor)
    else:
        raise AssertionError('unrecognized method')
    return pi_t



# -------------------------------------------------------------------------------------------

def model_based_learning(args, M, n_traj, gamma_guidance, l2_factor=None, l1_factor=None):
    if l2_factor is not None or l1_factor is not None:
        raise AssertionError('Not supported')
    nS = args.nS
    nA = args.nA
    epsilon = args.epsilon
    # Initial  behaviour policy - this is the policy used for collecting data
    if 'initial_policy' not in args or args.initial_policy == 'uniform':
        pi_b = GetUniformPolicy(nS, nA)
    elif args.initial_policy == 'generated_random':
        pi_b = draw_policy_at_random(nS, nA)
    else:
        raise AssertionError('Unrecognized args.initial_policy')

    pi_t = pi_b.copy()  # Initial  target policy  - the is the policy maintained by policy iteration
    data_history = []  # data from all previous episodes
    # Run episodes
    for i_episode in range(args.n_episodes):
        # Generate data:
        data = M.SampleData(args, pi_b, n_traj, p0=None)
        data_history += data
        # Improve policy:
        # 1. Estimate model:
        P_est, R_est = ModelEstimation(data_history, nS, nA)
        # 2. Certainty-Equivalence policy w.r.t model-estimation and gamma_guidance:
        pi_t, _, _ = PolicyIteration_GivenRP(R_est, P_est, gamma_guidance, args)
        pi_b = (1 - epsilon) * pi_t + (epsilon / nA)
    # end for i_episode
    return pi_t

# -------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------
def model_free_approx_policy_iteration(args, M, n_traj, gamma_guidance, l2_factor=None, l1_factor=None):
    nS = args.nS
    nA = args.nA
    data_history = []  # data from all previous episodes

    # Initial  behaviour policy - this is the policy used for collecting data
    if 'initial_policy' not in args or args.initial_policy == 'uniform':
        pi_b = GetUniformPolicy(nS, nA)
    elif args.initial_policy == 'generated_random':
        pi_b = draw_policy_at_random(nS, nA)
    else:
        raise AssertionError('Unrecognized args.initial_policy')

    pi_t = pi_b.copy()  # Initial  target policy  - the is the policy maintained by policy iteration
    Q_est = None  # the Q-function will be initialized in the first evaluation step

    # Run episodes:
    for i_episode in range(args.n_episodes):
        # Generate data:
        data = M.SampleData(args, pi_b,  n_traj, p0=None)
        data_history += data

        # Improve value estimation:
        if args.method in {'Expected_SARSA'}:
            # since this is off-policy evaluation - use all data
            Q_est = run_expected_sarsa(data_history, pi_t, gamma_guidance, args,
                                       initial_Q=Q_est, l2_factor=l2_factor, l1_factor=l1_factor)
        elif args.method in {'LSTDQ', 'ELSTDQ'}:
            # since this is off-policy evaluation - use all data
            Q_est = LSTDQ(data_history, pi_t, gamma_guidance, args, l2_factor=l2_factor, l1_factor=l1_factor)

        elif args.method in {'LSTDQ_nested', 'ELSTDQ_nested'}:
            # since this is off-policy evaluation - use all data
            Q_est = LSTDQ_nested(data_history, pi_t, gamma_guidance, args, l2_factor=l2_factor, l1_factor=l1_factor)

        elif args.method in {'SARSA'}:
            # since this is on-policy evaluation - use data only from current policy
            Q_est = run_sarsa(data, nS, nA, gamma_guidance, args, initial_Q=Q_est, l2_factor=l2_factor, l1_factor=l1_factor)
        else:
            raise AssertionError('unrecognized method')

        # Improve policy:
        pi_t = generalized_greedy(Q_est)

        #  behaviour policy : For exploration set an epsilon-greedy policy:
        epsilon = args.epsilon
        pi_b = (1 - epsilon) * pi_t + (epsilon / nA)
    return pi_t



# ------------------------------------------------------------------------------------------------------------~
# value estimation:
def run_value_estimation_method(data, M, args, gamma_guidance, l2_proj, l2_fp, l2_TD):
    gammaEval = args.gammaEval
    nS = M.nS

    V_true = np.linalg.solve((np.eye(nS) - gammaEval * M.P), M.R)

    if args.alg_type == 'LSTD':
        V_est = LSTD(data, gamma_guidance, args, l2_factor=l2_proj)

    elif args.alg_type == 'LSTD_Nested':
        V_est = LSTD_Nested(data, gamma_guidance, args, l2_proj, l2_fp)

    elif args.alg_type == 'LSTD_Nested_Standard':
        V_est = LSTD_Nested_Standard(data, gamma_guidance, args, l2_proj, l2_fp)

    elif args.alg_type == 'batch_TD_value_evaluation':
        V_est = batch_TD_value_evaluation(data, gamma_guidance, args, l2_factor=l2_TD)

    elif args.alg_type == 'model_based_pol_eval':
        V_est = model_based_pol_eval(data, gamma_guidance, args)

    elif args.alg_type == 'model_based_known_P':
        V_est = model_based_known_P(data, gamma_guidance, args, M)
    else:
        raise AssertionError('Unrecognized args.grid_type')

    return V_est, V_true
    # ------------------------------------------------------------------------------------------------------------~
def set_learning_rate(i_iter, args, gamma_guidance):
    learning_rate_def = args.learning_rate_def
    lr_type = learning_rate_def['type']
    if lr_type == 'const':
        alpha = learning_rate_def['alpha']
    elif lr_type == 'a/(b+i_iter)':
        a = learning_rate_def['a']
        b = learning_rate_def['b']
        alpha = a / (b + i_iter)
    elif args.learning_rate_def == 'a/(b+sqrt(i_iter))':
        a = learning_rate_def['a']
        b = learning_rate_def['b']
        alpha = a / (b + np.sqrt(i_iter))
    else:
        raise AssertionError('Invalid learning_rate_def')
    if learning_rate_def['scale']:
        # explanation to scaling:  according to equivalency proposition, this scaling would make discount reg (using gamma) to behave like using gammaEval but with added reg term.
        gammaEval = args.gammaEval
        alpha *= gammaEval / gamma_guidance
    return alpha

# ------------------------------------------------------------------------------------------------------------~
def set_initial_value(args, shape, gamma_guidance):
    TD_Init_type = args.TD_Init_type
    gammaEval = args.gammaEval
    Rmax = 1  # assumption in mdp creation
    Vmax = Rmax / (1-gammaEval)
    # explanation to scaling:  according to equivalency proposition, this scaling would make discount reg (using gamma) to behave like using gammaEval but with added reg term.
    VmaxGamma = Rmax / (1-gamma_guidance)
    if isinstance(shape, int):
        shape = [shape]
    if TD_Init_type == 'random_0_1':
        x = np.random.rand(*shape)
    elif TD_Init_type == 'random_0_Vmax':
        x = np.random.rand(*shape) * Vmax
    elif TD_Init_type == 'zero':
        x = np.zeros(shape)
    elif TD_Init_type == 'Vmax':
        x = np.ones(shape) * Vmax
    elif TD_Init_type == '0.5_Vmax':
        x = 0.5 * np.ones(shape) / (1 - gammaEval)
    elif TD_Init_type == 'VmaxGamma':
        x = np.ones(shape) * VmaxGamma
    else:
        raise AssertionError('Invalid TD_Init_type')
    return x
# ------------------------------------------------------------------------------------------------------------~
def ModelEstimation(data, nS, nA):
    """
    Maximum-Likelihood estimation of model based on data

    Parameters:
    data: list of n trajectories, each is a list of sequence of depth tuples (state, action, reward, next state)
    S: number of states
    A: number of actions

    Returns:
    P_est: [S x A x S] estimated transitions probabilities matrix  P_{s,a,s'}=P(s'|s,a)
    R_est: [S x A] estimated mean rewards matrix R
    """

    counts_sas = np.zeros((nS, nA, nS))
    counts_sa = np.zeros((nS, nA))
    R_est = np.zeros((nS, nA))
    P_est = np.zeros((nS, nA, nS))
    for traj in data:
        for sample in traj:
            (s, a, r, s_next, a_next) = sample
            counts_sa[s, a] += 1
            counts_sas[s, a, s_next] += 1
            R_est[s, a] += r

    for s in range(nS):
        for a in range(nA):
            if counts_sa[s, a] == 0:
                # if this state-action doesn't exist in data
                # Use default values:
                R_est[s, a] = 0.5
                P_est[s, a, :] = 1 / nS
            else:
                R_est[s, a] /= counts_sa[s, a]
                P_est[s, a, :] = counts_sas[s, a, :] / counts_sa[s, a]
    if np.any(np.abs(P_est.sum(axis=2) - 1) > 1e-5):
        raise RuntimeError('Transition Probability matrix not normalized!!')
    return P_est, R_est


# ------------------------------------------------------------------------------------------------------------~
def TD_value_evaluation(data, nS, nA, gamma, args):
    """
    Runs TD iterations on data set of samples from unknown policy to estimate the value function of this policy

    Parameters:
    data: list of n trajectories, each is a list of sequence of depth tuples (state, action, reward, next state)
    nS: number of states
    nA: number of actions
    gamma: Discount factor

    Returns:
    V_est: [S] The estimated value-function for a fixed policy pi, i,e. the the expected discounted return when following pi starting from some state
    """

    gammaEval = args.gammaEval
    # Initialization:
    V_est = set_initial_value(args, nS, gamma)

    # prev_V = V_pi.copy()
    # stop_diff = 1e-5  # stopping condition

    # Join list of data tuples from all trajectories:
    data_tuples = sum(data, [])
    n_samples = len(data_tuples)
    for i_iter in range(args.n_TD_iter):
        alpha = set_learning_rate(i_iter, args, gamma)
        # Choose random sample:
        i_sample = randrange(n_samples)
        (s, a, r, s_next, a_next) = data_tuples[i_sample]
        if args.use_reward_scaling:
            r *= gamma / gammaEval
        delta = r + gamma * V_est[s_next] - V_est[s]
        V_est[s] += alpha * delta
    # end for i_iter
        # if i_iter > 0 and (i_iter % 10000 == 0):
        #     if np.linalg.norm(V_pi - prev_V) < stop_diff:
        #         break
        #     prev_V = V_pi
    return V_est



# ------------------------------------------------------------------------------------------------------------~
def run_sarsa(data, nS, nA, gamma, args, initial_Q=None, l2_factor=None, l1_factor=None):
    """
      Runs TD iterations on data set of samples from unknown policy to estimate the Q-function of this policy
      using SARSA algorithm

    Parameters:
    data: list of n trajectories, each is a list of sequence of depth tuples (state, action, reward, next state)
    nS: number of states
    nA: number of actions
    gamma: Discount factor

    Returns:
    Q_est [S x A] The estimated Q-function for a fixed policy pi, i,e. the the expected discounted return when following pi starting from some state and action
    """
    gammaEval = args.gammaEval
    # Initialization:
    if initial_Q is None:
        Q_est = set_initial_value(args, (nS, nA), gamma)
    else:
        Q_est = initial_Q

    # prev_V = V_pi.copy()
    # stop_diff = 1e-5  # stopping condition

    # Join list of data tuples from all trajectories:
    data_tuples = sum(data, [])
    n_samples = len(data_tuples)
    for i_iter in range(args.n_TD_iter):
        alpha = set_learning_rate(i_iter, args, gamma)
        # Choose random sample:
        i_sample = randrange(n_samples)
        (s, a, r, s_next, a_next) = data_tuples[i_sample]
        if args.use_reward_scaling:
            r *= gamma / gammaEval
        delta = r + gamma * Q_est[s_next, a_next] - Q_est[s, a]
        Q_est[s, a] += alpha * delta

        # Add the gradient of the added regularization term:
        if l2_factor is not None:
            reg_grad = 2 * l2_factor * Q_est  # gradient of the L2 regularizer [tabular case]
            Q_est -= alpha * reg_grad

        if l1_factor is not None:
            reg_grad = 1 * l1_factor * np.sign(Q_est)  # gradient of the L2 regularizer [tabular case]
            Q_est -= alpha * reg_grad
    # end for i_iter

    return Q_est


# ------------------------------------------------------------------------------------------------------------~
def run_expected_sarsa(data, pi, gamma, args, initial_Q=None, l2_factor=None, l1_factor=None):
    """
      Runs TD iterations on data set of samples from given  policy to estimate the Q-function of this policy
       using Expected-SARSA algorithm

    Parameters:
    data: list of n trajectories, each is a list of sequence of depth tuples (state, action, reward, next state)
    pi: the policy that generated that generated the data
    S: number of states
    A: number of actions
    gamma: Discount factor

    Returns:
    Q_est [S x A] The estimated Q-function for the fixed policy pi, i,e. the the expected discounted return when following pi starting from some state and action
    """
    if pi.ndim != 2:
        raise AssertionError('Invalid input')
    S = pi.shape[0]
    A = pi.shape[1]
    gammaEval = args.gammaEval
    # Initialization:
    if initial_Q is None:
        Q_est = set_initial_value(args, (S, A), gamma)
    else:
        Q_est = initial_Q

    # prev_V = V_pi.copy()
    # stop_diff = 1e-5  # stopping condition

    # Join list of data tuples from all trajectories:
    data_tuples = sum(data, [])
    n_samples = len(data_tuples)
    for i_iter in range(args.n_TD_iter):
        alpha = set_learning_rate(i_iter, args, gamma)
        # Choose random sample:
        i_sample = randrange(n_samples)
        (s, a, r, s_next, a_next) = data_tuples[i_sample]
        if args.use_reward_scaling:
            r *= gamma / gammaEval
        V_next = np.dot(Q_est[s_next, :], pi[s_next, :])
        delta = r + gamma * V_next - Q_est[s, a]
        Q_est[s, a] += alpha * delta

        # Add the gradient of the added regularization term:
        if l2_factor is not None:
            reg_grad = 2 * l2_factor * Q_est  # gradient of the L2 regularizer [tabular case]
            Q_est -= alpha * reg_grad

        if l1_factor is not None:
            reg_grad = 1 * l1_factor * np.sign(Q_est)  # gradient of the L2 regularizer [tabular case]
            Q_est -= alpha * reg_grad
    # end for i_iter
    return Q_est
# ------------------------------------------------------------------------------------------------------------~


def LSTDQ(data, pi, gamma, args, l2_factor=None, l1_factor=None):
    """
       given  policy to estimate the Q-function of this policy
       using LSTDQ algorithm
       "Least-squares policy iteration, MG Lagoudakis, R Parr - Journal of machine learning research, 2003"

    Parameters:
    data: list of n trajectories, each is a list of sequence of depth tuples (state, action, reward, next state)
    pi: the policy that generated that generated the data
    S: number of states
    A: number of actions
    gamma: Discount factor

    Returns:
    Q_est [S x A] The estimated Q-function for the fixed policy pi, i,e. the the expected discounted return when following pi starting from some state and action
    """
    if l1_factor is not None:
        raise AssertionError('Not supported yet')

    if l2_factor is None:
        l2_factor = args.default_l2_factor  # to prevent  Singular matrix
    if pi.ndim != 2:
        raise AssertionError('Invalid input')
    nS = pi.shape[0]
    nA = pi.shape[1]
    gammaEval = args.gammaEval
    # Join list of data tuples from all trajectories:
    data_tuples = sum(data, [])

    n_samples = len(data_tuples)
    n_feat = nS * nA
    Amat = np.zeros((n_feat, n_feat))
    bmat = np.zeros((n_feat, 1))

    for i_samp in range(n_samples):
        (s, a, r, s_next, a_next) = data_tuples[i_samp]
        if args.use_reward_scaling:
            r *= gamma / gammaEval
            raise AssertionError('use_reward_scaling should be False with LSTDQ')
        ind1 = s_a_to_ind(s, a, nS, nA)
        Amat[ind1, ind1] += 1
        bmat[ind1] += r
        if args.method == 'LSTDQ':
            # SARSA style update
            ind2 = s_a_to_ind(s_next, a_next, nS, nA)
            Amat[ind1, ind2] -= gamma
        elif args.method == 'ELSTDQ':
            # Expected SARSA style update
            for a_prime in range(nA):
                ind2 = s_a_to_ind(s_next, a_prime, nS, nA)
                Amat[ind1, ind2] -= gamma * pi[s, a_prime]
        else:
            raise AssertionError

    Qest_vec = np.linalg.solve(Amat + l2_factor * np.eye(n_feat), bmat)
    Q_est = np.reshape(Qest_vec, (nS, nA))
    return Q_est
# ------------------------------------------------------------------------------------------------------------~


def s_a_to_ind(s, a, nS, nA):
    ind = s * nA + a
    return ind
# ------------------------------------------------------------------------------------------------------------~


def ind_to_s_a(ind, nS, nA):
    s = ind // nA
    a = ind % nA
    return s, a
# ------------------------------------------------------------------------------------------------------------~


def LSTDQ_nested(data, pi, gamma, args, l2_factor=None, l1_factor=None):
    """
       given  policy to estimate the Q-function of this policy
       using LSTDQ algorithm
       "Least-squares policy iteration, MG Lagoudakis, R Parr - Journal of machine learning research, 2003"
            based on:
     regularized least-squares temporal difference learning with nested ℓ 2 and ℓ 1 penalizationMW Hoffman, A Lazaric, M Ghavamzadeh, R Munos - European Workshop on …, 2011


    Parameters:
    data: list of n trajectories, each is a list of sequence of depth tuples (state, action, reward, next state)
    pi: the policy that generated that generated the data
    S: number of states
    A: number of actions
    gamma: Discount factor

    Returns:
    Q_est [S x A] The estimated Q-function for the fixed policy pi, i,e. the the expected discounted return when following pi starting from some state and action
    """
    if l1_factor is not None:
        raise AssertionError('Not supported yet')

    if l2_factor is None:
        l2_factor = args.default_l2_factor  # to prevent  Singular matrix
    if pi.ndim != 2:
        raise AssertionError('Invalid input')
    nS = pi.shape[0]
    nA = pi.shape[1]
    gammaEval = args.gammaEval
    # Join list of data tuples from all trajectories:
    data_tuples = sum(data, [])

    n_samples = len(data_tuples)
    n_feat = nS * nA + 1  # +1 for bias

    I = np.eye((n_feat))
    Phi = np.zeros((n_samples, n_feat))  # For each sample: feature of current state
    PhiPrime = np.zeros((n_samples, n_feat))   # For each sample: feature of next state
    R = np.zeros((n_samples, 1))  # For each sample: reward

    for i_samp in range(n_samples):
        (s, a, r, s_next, a_next) = data_tuples[i_samp]
        if args.use_reward_scaling:
            r *= gamma / gammaEval
        ind1 = s_a_to_ind(s, a, nS, nA)
        Phi[i_samp, ind1] = 1.
        Phi[i_samp, -1] = 1.  # for bias
        PhiPrime[i_samp, -1] = 1.  # for bias
        R[i_samp] = r

        if args.method == 'LSTDQ_nested':
            # SARSA style update
            ind2 = s_a_to_ind(s_next, a_next, nS, nA)
            PhiPrime[i_samp, ind2] = 1.
        elif args.method == 'ELSTDQ_nested':
            # Expected SARSA style update
            for a_prime in range(nA):
                ind2 = s_a_to_ind(s_next, a_prime, nS, nA)
                PhiPrime[i_samp, ind2] = pi[s, a_prime]
        else:
            raise AssertionError

    l2_proj = l2_fp = l2_factor
    PhiBar = Phi.mean(axis=0)  # features means
    PhiPrimeBar = PhiPrime.mean(axis=0)  # features means
    RBar = R.mean(axis=0)
    PhiTilde = Phi - PhiBar
    PhiPrimeTilde = PhiPrime - PhiPrimeBar
    Rtilde = R - RBar
    sigmaPhi = PhiTilde.std(axis=0)
    sigmaPhi[sigmaPhi == 0] = 1.
    PhiHat = PhiTilde / sigmaPhi
    SigmaMat = PhiHat @ np.linalg.inv(PhiHat.T @ PhiHat + l2_proj * np.eye(n_feat)) @ PhiHat.T
    Xmat = Phi - gamma * SigmaMat @ PhiPrimeTilde - gamma * matlib.repmat(PhiPrimeBar, n_samples, 1)
    yMat = SigmaMat @ Rtilde + matlib.repmat(RBar, n_samples, 1)
    Amat = Xmat.T @ Xmat + l2_fp * np.eye(n_feat)
    bmat = Xmat.T @ yMat
    theta_vec = np.linalg.solve(Amat, bmat)
    Qest_vec = theta_vec[:-1] + theta_vec[-1]  #  add bias term ... Q(s,a) = (phi.T @ theta)_{s,a} =  theta[s,a] + theta[-1]
    Q_est = np.reshape(Qest_vec, (nS, nA))
    return Q_est
# ------------------------------------------------------------------------------------------------------------~


def LSTD(data, gamma, args, l2_factor):
    """
       given  policy to estimate the Q-function of this policy
       using LSTDQ algorithm

    Parameters:
    data: list of n trajectories, each is a list of sequence of depth tuples (state, action, reward, next state)
    pi: the policy that generated that generated the data
    S: number of states
    A: number of actions
    gamma: Discount factor

    Returns:
    V_est [nS x 1] The estimated Q-function for the fixed policy pi, i,e. the the expected discounted return when following pi starting from some state and action
    """

    # Join list of data tuples from all trajectories:
    data_tuples = sum(data, [])

    nS = args.nS
    gammaEval = args.gammaEval
    n_samples = len(data_tuples)
    n_feat = nS
    Amat = np.zeros((n_feat, n_feat))
    bmat = np.zeros((n_feat, 1))

    for i_samp in range(n_samples):
        (s, r, s_next) = data_tuples[i_samp]
        if args.use_reward_scaling:
            r *= gamma / gammaEval
        Amat[s, s] += 1
        Amat[s, s_next] -= gamma
        bmat[s] += r

    Vest_vec = np.linalg.solve(Amat + l2_factor * np.eye(n_feat), bmat)
    V_est = np.reshape(Vest_vec, (nS, 1))
    return V_est

# ------------------------------------------------------------------------------------------------------------~
def LSTD_Nested(data, gamma, args, l2_proj=0., l2_fp=0.):
    """
     based on:
     regularized least-squares temporal difference learning with nested ℓ 2 and ℓ 1 penalizationMW Hoffman, A Lazaric, M Ghavamzadeh, R Munos - European Workshop on …, 2011

    Parameters:
    data: list of n trajectories, each is a list of sequence of depth tuples (state, action, reward, next state)
    pi: the policy that generated that generated the data
    S: number of states
    A: number of actions
    gamma: Discount factor

    Returns:
    V_est [nS x 1] The estimated Q-function for the fixed policy pi, i,e. the the expected discounted return when following pi starting from some state and action
    """

    # Join list of data tuples from all trajectories:
    data_tuples = sum(data, [])

    nS = args.nS
    gammaEval = args.gammaEval
    n_samples = len(data_tuples)
    n_feat = nS
    Amat = np.zeros((n_feat, n_feat))
    bmat = np.zeros((n_feat, 1))
    I = np.eye((n_feat))
    Phi = I # tabular case

    PhiTPhi = np.zeros((n_feat, n_feat))
    PhiTPhiPrime = np.zeros((n_feat, n_feat))

    for i_samp in range(n_samples):
        (s, r, s_next) = data_tuples[i_samp]
        if args.use_reward_scaling:
            r *= gamma / gammaEval
        PhiTPhi[s, s] += 1
        PhiTPhiPrime[s, s_next] += 1
        Amat[s, s] += 1
        Amat[s, s_next] -= gamma
        bmat[s] += r

    # Amat = PhiTPhi - gamma * PhiTPhiPrime
    Cmat = Phi @ np.linalg.inv(PhiTPhi + l2_proj * I)
    Xmat = Cmat @ (Amat + l2_proj * I)
    yMat = Cmat @ bmat
    theta_vec =  np.linalg.solve(Xmat.T @ Xmat + l2_fp * I,  Xmat.T @ yMat)

    V_est = np.reshape(theta_vec, (nS, 1))
    return V_est

# ------------------------------------------------------------------------------------------------------------~
def LSTD_Nested_Standard(data, gamma, args, l2_proj=0., l2_fp=0.):
    """
     based on:
     regularized least-squares temporal difference learning with nested ℓ 2 and ℓ 1 penalizationMW Hoffman, A Lazaric, M Ghavamzadeh, R Munos - European Workshop on …, 2011

    Parameters:
    data: list of n trajectories, each is a list of sequence of depth tuples (state, action, reward, next state)
    pi: the policy that generated that generated the data
    S: number of states
    A: number of actions
    gamma: Discount factor

    Returns:
    V_est [nS x 1] The estimated Q-function for the fixed policy pi, i,e. the the expected discounted return when following pi starting from some state and action
    """

    # Join list of data tuples from all trajectories:
    data_tuples = sum(data, [])

    nS = args.nS
    gammaEval = args.gammaEval
    n_samples = len(data_tuples)
    n_feat = nS + 1  # +1 for bias

    I = np.eye((n_feat))

    Phi = np.zeros((n_samples, n_feat))  # For each sample: feature of current state
    PhiPrime = np.zeros((n_samples, n_feat))   # For each sample: feature of next state
    R = np.zeros((n_samples, 1))  # For each sample: reward

    for i_samp in range(n_samples):
        (s, r, s_next) = data_tuples[i_samp]
        if args.use_reward_scaling:
            r *= gamma / gammaEval
        Phi[i_samp, s] = 1.
        Phi[i_samp, -1] = 1.  # for bias
        PhiPrime[i_samp, s_next] = 1.
        PhiPrime[i_samp, -1] = 1.  # for bias
        R[i_samp] = r

    PhiBar = Phi.mean(axis=0)  # features means
    PhiPrimeBar = PhiPrime.mean(axis=0)  # features means
    RBar = R.mean(axis=0)


    PhiTilde = Phi - PhiBar
    PhiPrimeTilde = PhiPrime - PhiPrimeBar
    Rtilde = R - RBar

    sigmaPhi = PhiTilde.std(axis=0)
    sigmaPhi[sigmaPhi == 0] = 1.

    PhiHat = PhiTilde / sigmaPhi

    SigmaMat = PhiHat @ np.linalg.inv(PhiHat.T @ PhiHat + l2_proj * np.eye(n_feat)) @ PhiHat.T
    Xmat = Phi - gamma * SigmaMat @ PhiPrimeTilde - gamma * matlib.repmat(PhiPrimeBar, n_samples, 1)
    yMat = SigmaMat @ Rtilde + matlib.repmat(RBar, n_samples, 1)
    Amat = Xmat.T @ Xmat + l2_fp * np.eye(n_feat)
    bmat = Xmat.T @ yMat
    theta_vec = np.linalg.solve(Amat, bmat)
    V_est = theta_vec[:-1] + theta_vec[-1]  #  add bias term ... V(s) = (phi.T @ theta)_s =  theta[s] + theta[-1]

    return V_est

# ------------------------------------------------------------------------------------------------------------~

# # ------------------------------------------------------------------------------------------------------------~
def batch_TD_value_evaluation(data, gamma, args, l2_factor=None):
    """
    Runs TD iterations on data set of samples from unknown policy to estimate the value function of this policy

    Parameters:
    data: list of n trajectories, each is a list of sequence of depth tuples (state, action, reward, next state)
    gamma: Discount factor

    Returns:
    V_est: [S] The estimated value-function for a fixed policy pi, i,e. the the expected discounted return when following pi starting from some state
    """
    nS = args.nS
    gammaEval = args.gammaEval
    # Initialization:
    V_est = set_initial_value(args, nS, gamma)

    # Join list of data tuples from all trajectories:
    data_tuples = sum(data, [])

    n_samples = len(data_tuples)

    for i_iter in range(args.n_TD_iter):
        alpha = set_learning_rate(i_iter, args, gamma)
        # Choose random sample:
        i_sample = randrange(n_samples)
        (s, r, s_next) = data_tuples[i_sample]
        if args.use_reward_scaling:
            r *= gamma / gammaEval
        delta = r + gamma * V_est[s_next] - V_est[s]
        V_est[s] += alpha * delta
        # Add the gradient of the added regularization term:
        if l2_factor is not None:
            reg_grad = 2 * l2_factor * V_est  # gradient of the L2 regularizer [tabular case]
            V_est -= alpha * reg_grad
        # end if
    # end for i_iter
    return V_est
# # ------------------------------------------------------------------------------------------------------------~
# ------------------------------------------------------------------------------------------------------------~


def ModelEstimation_MRP(data, args):
    """
    Maximum-Likelihood estimation of model based on data

    Parameters:
    data: list of n trajectories, each is a list of sequence of depth tuples (state, action, reward, next state)
    S: number of states
    A: number of actions

    Returns:
    P_est: [S x A x S] estimated transitions probabilities matrix  P_{s,a,s'}=P(s'|s,a)
    R_est: [S x A] estimated mean rewards matrix R
    """
    nS = args.nS
    counts_ss = np.zeros((nS, nS))
    counts_s = np.zeros((nS))
    R_est = np.zeros((nS))
    P_est = np.zeros((nS, nS))
    for traj in data:
        for sample in traj:
            (s, r, s_next) = sample
            counts_s[s] += 1
            counts_ss[s, s_next] += 1
            R_est[s] += r
        # end for sample
    # end for sample

    for s in range(nS):
        if counts_s[s] == 0:
            # if this state-action doesn't exist in data
            # Use default values:
            R_est[s] = 0.5
            P_est[s, :] = 1 / nS
        else:
            R_est[s] /= counts_s[s]
            P_est[s, :] = counts_ss[s, :] / counts_s[s]
        # end if
    # end for s
    if np.any(np.abs(P_est.sum(axis=1) - 1) > 1e-5):
        raise RuntimeError('Transition Probability matrix not normalized!!')
    return P_est, R_est
# # ------------------------------------------------------------------------------------------------------------~


def model_based_pol_eval(data, gamma, args, l2_factor=None):
    if l2_factor is not None:
        raise AssertionError('Not supported yet')
    nS = args.nS
    P_est, R_est = ModelEstimation_MRP(data, args)
    V_est = np.linalg.solve((np.eye(nS) - gamma * P_est), R_est)
    return V_est
# # ------------------------------------------------------------------------------------------------------------~


def model_based_known_P(data, gamma, args, M, l2_factor=None):
    if l2_factor is not None:
        raise AssertionError('Not supported yet')
    nS = args.nS
    P = M.P #  get known P
    _, R_est = ModelEstimation_MRP(data, args)
    V_est = np.linalg.solve((np.eye(nS) - gamma * P), R_est)
    return V_est
# # ------------------------------------------------------------------------------------------------------------~


In [ ]:
# ===== File: utils/jobs_utils.py =====
import sys, os
project_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), '.'))
sys.path.insert(0, project_dir)


import glob, pickle
import numpy as np

from utils.common_utils import load_run_data, write_to_log, get_grid


#------------------------------------------------------------------------------------------------------------~

def get_num_finished(result_arr):
	if result_arr.ndim == 2:
		result_arr = result_arr[0]
	for i, x in enumerate(result_arr):
		if np.isnan(x):
			return i
	return result_arr.size


#------------------------------------------------------------------------------------------------------------~


def print_status(result_dir_to_load):

	args, info_dict = load_run_data(result_dir_to_load)
	alg_param_grid = get_grid(args.param_grid_def)
	n_grid = len(alg_param_grid)
	n_reps = args.n_reps
	print('Loaded parameters: \n', args, '\n', '-' * 20)
	if 'result_reward_mat' in info_dict.keys():
		result_reward_mat = info_dict['result_reward_mat']
	else:
		result_reward_mat = np.full(shape=(n_grid, n_reps), fill_value=np.nan)
	path = os.path.join(result_dir_to_load, 'jobs')
	all_files = glob.glob(os.path.join(path, "*.p"))
	n_rep_finish_per_point = np.full(shape=n_grid, fill_value=0, dtype=np.int)
	n_steps_finished = np.full(shape=(n_grid, n_reps), fill_value=-1, dtype=np.int)
	for f_path in all_files:
		save_dict = pickle.load(open(f_path, "rb"))
		job_info = save_dict['job_info']
		timesteps_snapshots = save_dict['timesteps_snapshots']
		i_grid = np.searchsorted(alg_param_grid, job_info['grid_param'], 'right')
		if i_grid == len(alg_param_grid):
			# the loaded param is not in our defined alg_param_grid
			continue
			# TODO: add option to load all files, even if not in the defined alg_param_grid (show_all_saved_results flag)
		i_rep = job_info['i_rep']
		if not np.isnan(result_reward_mat[i_grid, i_rep]):
			n_steps_finished[i_grid, i_rep] = args.max_timesteps
			n_rep_finish_per_point[i_grid] += 1
		else:
			n_steps_finished[i_grid, i_rep] = timesteps_snapshots[-1]

	for i_grid, grid_param in enumerate(alg_param_grid):
		write_to_log('Grid point {}/{}, val: {}, Number of finished reps loaded: {}'.
					 format(1 + i_grid, len(alg_param_grid), grid_param, n_rep_finish_per_point[i_grid]), args)
		for i_rep in range(n_reps):
			if n_steps_finished[i_grid, i_rep] != -1:
				print('Rep: {}, Finished Time-Steps: {}'.format(i_rep, n_steps_finished[i_grid, i_rep]))

## 2) Esperimenti tabulari e script principali

Ogni cella sotto corrisponde a uno script originale. Eseguendo la cella lanci l’esperimento così com’è (puoi modificare i parametri direttamente nel codice della cella).


In [ ]:
# ===== File: main_MRP.py =====
import numpy as np
import matplotlib.pyplot as plt
import argparse
import timeit
import time
import ray
from copy import deepcopy


set_default_plot_params()

# -------------------------------------------------------------------------------------------
#  Run mode
# -------------------------------------------------------------------------------------------

local_mode = False  # True/False - run non-parallel to get error messages and debugging
# Option to load previous run results:
load_run = False  # False/True If true just load results from dir, o.w run simulationz
result_dir_to_load = './saved/main_MRP/2020_06_16_15_31_31'
# result_dir_to_load = './saved/Multi_Exp/2020_05_30_07_40_26/Exp_6_LSTD_Discount_MixTime'
# result_dir_to_load = './saved/Tabular/2020_06_16_00_52_40_PolEval_LSTD_L2Loss_L2Reg'

# Plot options:
save_PDF = False  # False/True - save figures as PDF file in the result folder
y_lim = None # [75, 85] | None
legend_loc = 'best' # | 'best'| 'upper left'
show_stars = True
# -------------------------------------------------------------------------------------------
#  Set Parameters
# -------------------------------------------------------------------------------------------

args = argparse.Namespace()

# ----- Run Parameters ---------------------------------------------#
args.run_name = ''   # 'Name of dir to save results in (if empty, name by time)'
args.seed = 1  # random seed
args.n_reps = 1000  # default 5000  # number of experiment repetitions

#  how to create parameter grid:
# args.param_grid_def = {'type': 'gamma_guidance', 'spacing': 'linspace', 'start': 0.4, 'stop': 0.99, 'num': 50, 'decimals': 10}
# args.param_grid_def = {'type': 'l2_factor', 'spacing': 'linspace', 'start': 0., 'stop': 0.9, 'num': 50, 'decimals': 10}
# args.param_grid_def = {'type': 'l2_fp', 'spacing': 'linspace', 'start': 0.0001, 'stop': 0.1, 'num': 50, 'decimals': 10} # $L_2$ Regularization Factor  - Fixed-Point Phase
# args.param_grid_ def = {'type': 'l2_proj', 'spacing': 'linspace', 'start': 0.0001, 'stop': 0.001, 'num': 20, 'decimals': 10} $L_2$ Regularization Factor  - Projection Phase
# args.param_grid_def = {'type': 'n_trajectories', 'spacing': 'arange', 'start': 1, 'stop': 21}
# args.param_grid_def = {'type': 'depth', 'spacing': 'arange', 'start': 1, 'stop': 10}
args.param_grid_def = {'type': 'states_TV_dist_from_uniform', 'spacing': 'list', 'list': [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]}
#
# ----- Problem Parameters ---------------------------------------------#


# args.mrp_def = {'type': 'ToyMRP', 'p01': 0.5, 'p10': 0.5,  'reward_std': 0.1}
# args.mrp_def = {'type': 'Chain', 'p_left': 0.5, 'length': 9,  'mean_reward_range': (0, 1), 'reward_std': 0.1}
# args.mrp_def = {'type': 'RandomMDP', 'S': 100, 'A': 2, 'k': 5, 'reward_std': 0.1, 'policy': 'uniform'}
args.mrp_def = {'type': 'GridWorld', 'N0': 4, 'N1': 4, 'reward_std': 0.5, 'forward_prob_distrb': 'uniform', 'goal_reward': 1, 'R_low': -0.5, 'R_high': 0.5, 'policy': 'uniform'}

# args.config_grid_def = {'type': 'p_left', 'spacing': 'list', 'list': [0.3, 0.5, 0.7, 0.9]} # for mrp_def['type'] == 'Chain':
# args.config_grid_def = {'type': 'forced_mix_time', 'spacing': 'list', 'list': [1.5, 3, 6, 12]} # must be above  1.0
# args.config_grid_def = {'type': 'n_trajectories', 'spacing': 'list', 'list': [1, 2, 4, 8]}
# args.config_grid_def = {'type': 'trajectory_len', 'spacing': 'list', 'list': [10, 20, 30]}
# args.config_grid_def = {'type': 'states_TV_dist_from_uniform', 'spacing': 'list', 'list': [0, 0.2, 0.4, 0.6, 0.8]}   # note: above 0.9 is very slow
# args.config_grid_def = {'type': 'None', 'spacing': 'list', 'list': [None]}
args.config_grid_def = {'type': 'RegMethod', 'spacing': 'list_tuples',  'list':[ ('None', None), ('Discount Reg.', 0.98), ('L2 Reg.', 0.17)]} #

args.depth = 50  # default: 10 for 'chain', 100 for 'GridWorld'  # Length of trajectory
args.gammaEval = 0.99   # default: 0.99  # gammaEval

args.train_sampling_def = {'type': 'Trajectories'}
# args.train_sampling_def = {'type': 'Generative_uniform'}
# args.train_sampling_def = {'type': 'sample_all_s'}



args.evaluation_loss_type = 'L2_uni_weight' #  'rankings_kendalltau' | 'L2_uni_weight | 'L2' | 'one_pol_iter_l2_loss'

args.initial_state_distrb_type = 'uniform'  # 'uniform' | 'middle'
args.n_trajectories = 8  #

# ----- Algorithm Parameters ---------------------------------------------#
args.default_gamma = None  # default: None  # The default guidance discount factor (if None use gammaEval)

args.alg_type = 'LSTD'  # 'LSTD' | 'LSTD_Nested' | 'batch_TD_value_evaluation' | 'LSTD_Nested_Standard' | 'model_based_pol_eval' | 'model_based_known_P'
args.use_reward_scaling = False  # False | True.  set False for LSTD

# args.base_lstd_l2_fp = 1e-5
args.base_lstd_l2_proj = 1e-4

# if batch_TD_value_evaluation is used:
args.default_l2_TD = None  # default: None  # The default L2 factor for TD (if using discount regularization)
# args.TD_Init_type = 'zero'  # How to initialize V # Options: 'Vmax' | 'zero' | 'random_0_1' |  'random_0_Vmax' | '0.5_'Vmax' |
# args.n_TD_iter = 5000  # Default: 500 for RandomMDP, 5000 for GridWorld  # number of TD iterations
# args.learning_rate_def = {'type': 'a/(b+i_iter)', 'a': 500, 'b': 1000, 'scale': False}
# -------------------------------------------------------------------------------------------

def set_config(args, config_val):
	config_type = args.config_grid_def['type']
	if config_type == 'n_trajectories':
		args.n_trajectories = config_val

	elif config_type == 'trajectory_len':
		args.depth = config_val

	elif config_type == 'p_left':
		args.mrp_def['p_left'] = config_val

	elif config_type == 'forced_mix_time':
		args.forced_mix_time = config_val

	elif config_type == 'states_TV_dist_from_uniform':
		args.train_sampling_def = {'type': 'Generative', 'states_TV_dist_from_uniform': config_val}

	elif config_type == 'None':
		pass

	elif config_type == 'RegMethod':
		reg_str = config_val[0]
		reg_val = config_val[1]
		if reg_str == 'Discount Reg.':
			args.default_gamma = reg_val
		elif reg_str == 'L2 Reg.':
			args.default_l2_TD = reg_val
		elif reg_str == 'None':
			pass
		else:
			raise AssertionError
		# end if
	else:
		raise AssertionError('Unrecognized config_grid_def type')
	# end if

	return args
# # -------------------------------------------------------------------------------------------


def set_params(args_r, param_val):
	gammaEval = args_r.gammaEval
	n_trajectories = args_r.n_trajectories
	if args_r.default_gamma is None:
		gamma_guidance = gammaEval
	else:
		gamma_guidance = args_r.default_gamma
	l2_TD = args_r.default_l2_TD

	alg_type = args_r.alg_type
	l2_proj = 0
	l2_fp = 0
	if alg_type in {'LSTD'}:
		l2_proj = args_r.base_lstd_l2_proj
	elif alg_type in {'LSTD_Nested', 'LSTD_Nested_Standard'}:
		l2_fp = args_r.base_lstd_l2_fp
		l2_proj = args_r.base_lstd_l2_proj
	elif alg_type in {'batch_TD_value_evaluation'}:
		l2_TD = args_r.default_l2_TD
	else:
		raise AssertionError
	regenerate_mrp = False

	grid_type = args_r.param_grid_def['type']

	if grid_type == 'gamma_guidance':
		gamma_guidance = param_val

	elif grid_type == 'l2_factor':
		l2_proj += param_val
		l2_TD = param_val
		l2_fp += param_val

	elif grid_type == 'l2_proj':
		l2_proj = args_r.base_lstd_l2_proj + param_val

	elif grid_type == 'l2_fp':
		l2_fp = args_r.base_lstd_l2_fp + param_val

	elif grid_type == 'n_trajectories':
		args_r.n_trajectories = param_val

	elif grid_type == 'states_TV_dist_from_uniform':
		regenerate_mrp = True
		args_r.train_sampling_def = {'type': 'Generative', 'states_TV_dist_from_uniform': param_val}

	elif grid_type == 'NoReg':
		pass
	# elif args.param_grid_def['type'] == 'depth':
	# 	depth = param_val
	else:
		raise AssertionError
	return regenerate_mrp, gamma_guidance, l2_TD, l2_fp, l2_proj
# # -------------------------------------------------------------------------------------------


# A Ray remote function.
# runs a single repetition of the experiment
@ray.remote  # (num_cpus=0.2)  # specify how much resources the process needs
def run_rep(i_rep, param_val_grid, config_grid, args):
	nS = args.nS

	n_grid = param_val_grid.shape[0]
	n_configs = args.n_configs
	loss_rep = np.zeros((n_configs, n_grid))

	# default values
	gammaEval = args.gammaEval

	for i_config, config_val in enumerate(config_grid):  # grid of n_configs
		args = set_config(args, config_val)

		# Generate MRP:
		M = MRP(args)

		for i_grid, param_val in enumerate(param_val_grid):
			# grid values:
			regenerate_mrp, gamma_guidance, l2_TD, l2_fp, l2_proj  = set_params(args, param_val)

			if regenerate_mrp:
				M = MRP(args)

			# Generate data:
			data = M.SampleDataMrp(args)

			V_est, V_true = run_value_estimation_method(data, M, args, gamma_guidance, l2_proj, l2_fp, l2_TD)

			loss_type = args.evaluation_loss_type
			pi = None
			eval_loss = evaluate_value_estimation(loss_type, V_true, V_est, M, pi, gammaEval, gamma_guidance)
			loss_rep[i_config, i_grid] = eval_loss
		# end for i_grid
	#  end for i_config
	return loss_rep
# end run_rep
# -------------------------------------------------------------------------------------------


def run_simulations(args, save_result, local_mode):
	args_def = deepcopy(args)
	start_ray(local_mode)
	if save_result:
		create_result_dir(args)
		write_to_log('local_mode == {}'.format(local_mode), args)

	start_time = timeit.default_timer()
	set_random_seed(args.seed)

	n_reps = args.n_reps
	param_val_grid = get_grid(args.param_grid_def)
	n_grid = param_val_grid.shape[0]

	config_grid = get_grid(args.config_grid_def)
	n_configs = len(config_grid)
	args.n_configs = n_configs

	loss_mat = np.zeros((n_reps, n_configs, n_grid))

	# ----- Run simulation in parrnell process---------------------------------------------#
	loss_rep_id_lst = []
	for i_rep in range(n_reps):
		# returns objects ids:
		loss_mat_rep_id = run_rep.remote(i_rep, param_val_grid, config_grid, args)
		loss_rep_id_lst.append(loss_mat_rep_id)
	# -----  get the results --------------------------------------------#
	for i_rep in range(n_reps):
		loss_rep = ray.get(loss_rep_id_lst[i_rep])
		write_to_log('Finished: {} out of {} reps'.format(i_rep + 1, n_reps), args)
		loss_mat[i_rep] = loss_rep
	# end for i_rep
	info_dict = {'loss_avg': loss_mat.mean(axis=0), 'loss_std': loss_mat.std(axis=0),
				 'param_val_grid': param_val_grid, 'config_grid': config_grid}
	if save_result:
		save_run_data(args, info_dict)
	stop_time = timeit.default_timer()
	write_to_log('Total runtime: ' +
				 time.strftime("%H hours, %M minutes and %S seconds", time.gmtime(stop_time - start_time)), args)
	write_to_log(['-'*10 +'Defined args: ', pretty_print_args(args_def), '-'*20], args)
	return info_dict
# end  run_simulations
# -------------------------------------------------------------------------------------------


def run_main_mrp(args, save_result=True,  local_mode=local_mode, load_run_data_flag=False, result_dir_to_load='',
                 save_PDF=False, plot=True, show_stars=True, y_lim=None, legend_loc='best'):
	SetMrpArgs(args)
	if load_run_data_flag:
		args, info_dict = load_run_data(result_dir_to_load)
	else:
		info_dict = run_simulations(args, save_result, local_mode)
	if 'loss_avg' in info_dict:
		loss_avg = info_dict['loss_avg']
		loss_std = info_dict['loss_std']
	else:
		loss_avg = info_dict['planing_loss_avg']
		loss_std = info_dict['planing_loss_std']
	if 'param_val_grid' in info_dict:
		param_val_grid = info_dict['param_val_grid']
	else:
		param_val_grid = info_dict['alg_param_grid']
	config_grid =  info_dict['config_grid']
	n_reps = args.n_reps

	# ----- Plot figures  ---------------------------------------------#
	if plot or save_PDF:
		ax = plt.figure().gca()
		if args.train_sampling_def['type'] in {'Generative', 'Generative_uniform', 'Generative_Stationary'}:
			data_size_per_traj = args.depth
		elif args.train_sampling_def['type'] == 'Trajectories':
			data_size_per_traj = args.depth
		elif args.train_sampling_def['type'] == 'sample_all_s':
			data_size_per_traj = args.nS
		else:
			raise AssertionError
		# end if
		xscale = 1.
		x_label = ''
		legend_title = ''
		grid_type = args.param_grid_def['type']
		if grid_type == 'gamma_guidance':
			x_label = r'Guidance Discount Factor $\gamma$'
		elif grid_type == 'l2_TD':
			x_label = r'$L_2$ Regularization Factor [1e-3]'
			xscale = 1e3
		elif grid_type == 'l2_proj':
			x_label = r'$L_2$ Regularization Factor  - Projection Phase'
		elif grid_type == 'l2_fp':
			x_label = r'$L_2$ Regularization Factor  - Fixed-Point Phase'
		elif grid_type == 'l2_TD':
			x_label = r'$L_2$ Regularization Factor'
		elif grid_type == 'l2_factor':
			x_label = r'$L_2$ Regularization Factor'
		elif grid_type == 'n_trajectories':
			if args.train_sampling_def['type'] in {'Generative', 'Generative_uniform'} \
					or args.config_grid_def['type'] == 'states_TV_dist_from_uniform':
				xscale = args.depth
				x_label = 'Num. Samples'
			else:
				x_label = 'Num. Trajectories'
			# end if
		elif grid_type == 'states_TV_dist_from_uniform':
			x_label = 'Total-Variation from\n uniform [normalized]'
		# end if
		ci_factor = 1.96 / np.sqrt(n_reps)  # 95% confidence interval factor
		# mixing_time_labels = ['Fast mixing', 'Moderate mixing',  'Slow mixing']
		for i_config, config_val in enumerate(config_grid): # for of plots in the figure

			config_type = args.config_grid_def['type']

			if config_type == 'p_left':
				args.mrp_def['p_left'] = config_val
				M = MRP(args)
				mixing_time = np.around(calc_typical_mixing_time(M.P), decimals=3)
				label_str = r'$\tau$={}'.format(mixing_time)
				print(label_str + r': $p_l=${} , $1/(1-|\lambda_2|)${}'.format(config_val, mixing_time))

			elif config_type == 'forced_mix_time':
				label_str = '{}'.format(config_val)
				legend_title = 'Mixing-time'

			elif config_type == 'n_trajectories':
				if args.train_sampling_def['type'] in {'Generative_uniform', 'Generative', 'Generative_Stationary'}:
					legend_title = 'Num. Samples'
					label_str = '{} '.format(config_val * data_size_per_traj)
				else:
					legend_title = 'Num. Trajectories'
					label_str = '{} '.format(config_val)

			elif config_type == 'states_TV_dist_from_uniform':
				legend_title = 'Total-Variation from\n uniform [normalized]'
				label_str = '{} '.format(config_val)

			elif config_type == 'None':
				legend_title = None
				label_str = ''

			elif config_type == 'RegMethod':
				legend_title = 'Regularizer'
				label_str = str(config_val[0]) + ': '+ str(config_val[1])


			else:
				raise AssertionError


			plt.errorbar(xscale * param_val_grid, loss_avg[i_config], yerr=loss_std[i_config] * ci_factor,
						 marker='.', label=label_str)

			if show_stars:
				# Mark the lowest point:
				i_best = np.argmin(loss_avg[i_config])
				plt.scatter(xscale * param_val_grid[i_best], loss_avg[i_config][i_best], marker='*', s=400)
				print(label_str + ' Best x-coord: ' + str(xscale *  param_val_grid[i_best]) )
			# end if
		# for i_config

		plt.grid(True)
		plt.ylabel(r'Loss')
		plt.xlabel(x_label)
		if args.evaluation_loss_type == 'L2':
			plt.ylabel(r'$L_2$ Loss')
		if args.evaluation_loss_type == 'L2_uni_weight':
			plt.ylabel(r'Avg. $L_{2}$ Loss')
		elif args.evaluation_loss_type == 'rankings_kendalltau':
			plt.ylabel(r'Ranking Loss')
		if legend_title is not None:
			plt.legend(title=legend_title, loc=legend_loc, fontsize=12, title_fontsize=12)  # loc='upper right'
		# plt.xlim([0.95,0.99])

		if y_lim:
			plt.ylim(y_lim)
		# ax.set_yticks(np.arange(0., 9., step=1.))

		if save_PDF:
			save_fig(args.run_name)
		else:
			# plt.title('Loss +- 95% CI \n ' + str(args.args))
			plt.title(args.mdp_def['type'] + '  ' + args.run_name + ' \n ' + args.result_dir, fontsize=6)
		# end if save_PDF
	# end if
	pretty_print_args(args)
	if plot:
		plt.show()
	info_dict['result_dir'] = args.result_dir
	return info_dict
# end  run_main
# -------------------------------------------------------------------------------------------

# -------------------------------------------------------------------------------------------
if __name__ == "__main__":
	info_dict = run_main_mrp(args, save_result=True, local_mode=local_mode, load_run_data_flag=load_run,
	                         result_dir_to_load=result_dir_to_load, save_PDF=save_PDF, y_lim=y_lim, legend_loc=legend_loc)
	# write_to_log(['Defined args: ', pretty_print_args(args_def)], info_dict['args'])
	print('done')





In [ ]:
# ===== File: main_control.py =====
"""
In  tabular MDP setting, evaluates the learning of optimal policy using different guidance discount factors
On-policy means we run episodes,
 in each episode we generate roll-outs/trajectories of current policy and run algorithm to improve the policy.
"""

import numpy as np
import matplotlib.pyplot as plt
import argparse
import timeit
import time
import ray
from copy import deepcopy
	start_ray, set_default_plot_params, save_fig, pretty_print_args

set_default_plot_params()

# -------------------------------------------------------------------------------------------
#  Run mode
# -------------------------------------------------------------------------------------------

local_mode = False  # True/False - run non-parallel to get error messages and debugging
# Option to load previous run results:
load_run = False  # False/True If true just load results from dir, o.w run simulation
# result_dir_to_load = './saved/Multi_Exp/2020_05_30_07_40_26/Exp_5_LSTDQ_Discount_TV_dist'
result_dir_to_load = './saved/main_control/2020_06_28_11_59_58/'


# Plot options:
save_PDF = False  # False/True - save figures as PDF file in the result folder
y_lim = None # [75, 85] | None
show_stars = True
# -------------------------------------------------------------------------------------------
#  Set Parameters
# -------------------------------------------------------------------------------------------

args = argparse.Namespace()

# -----  Run Parameters ---------------------------------------------#

args.run_name = ''  # 'Name of dir to save results in (if empty, name by time)'
args.seed = 1  # random seed

args.n_reps = 1000  # default 1000  # number of experiment repetitions

#  how to create parameter grid:
# args.param_grid_def = {'type': 'gamma_guidance', 'spacing': 'linspace', 'start': 0.5, 'stop': 0.99, 'num': 50, 'decimals': 10}
args.param_grid_def = {'type': 'L2_factor', 'spacing': 'linspace', 'start': 1e-4, 'stop': 0.01, 'num': 50, 'decimals': 10}


args.config_grid_def = {'type': 'n_trajectories', 'spacing': 'list', 'list': [4, 8, 16, 32]}  # grid of number of trajectories to generate per episode  Default:  [1, 2, 4, 8]
# args.config_grid_def = {'type': 'states_actions_TV_dist_from_uniform', 'spacing': 'list', 'list': [0, 0.3, 0.6, 0.9]}   # note: above 0.9 is very slow
# args.config_grid_def = {'type': 'chain_mix_time', 'spacing': 'list', 'list': [5, 10, 20, 40, 80]}  # must be above  1.0,  generates states and actions from a Markov chain with some mixing time
# args.config_grid_def = {'type': 'n_episodes', 'spacing': 'list', 'list': [1, 2, 3, 4]}
# args.config_grid_def = {'type': 'None', 'spacing': 'list', 'list': [None]}

# ----- Problem Parameters ---------------------------------------------#
# MDP definition ( see data_utils.SetMdpArgs)
# args.mdp_def = {'type': 'RandomMDP', 'S': 10, 'A': 5, 'k': 2, 'reward_std': 0.1}
args.mdp_def = {'type': 'GridWorld', 'N0': 4, 'N1': 4, 'reward_std': 0.1, 'forward_prob_distrb': 'uniform',  'goal_reward': 1, 'R_low': -0.5, 'R_high': 0.5}
# args.mdp_def = {'type': 'GridWorld', 'N0': 4, 'N1': 4, 'reward_std': 0.1, 'forward_prob_distrb': {'alpha': 3, 'beta': 1}, 'goal_reward': 1}


args.depth = 10  # default: 10  # Length of trajectory
args.n_trajectories = 4  # default value
args.gammaEval = 0.99  # default: 0.99  # gammaEval
args.n_episodes = 5  # Number of episodes

args.train_sampling_def = {'type': 'Trajectories'}  # n_traj  trajectories of length args.depth
# args.train_sampling_def = {'type': 'Generative_uniform'}  # n_samples =  args.depth X n_traj
# args.train_sampling_def = {'type': 'Generative_Stationary'}  #  n_samples =  args.depth X n_traj
# args.train_sampling_def = {'type': 'sample_all_s_a'}  # n_samples per (s,a) =  n_traj
# args.train_sampling_def = {'type': 'Generative', 'states_TV_dist_from_uniform': 0, 'actions_TV_dist_from_uniform': 0}  # n_samples =  args.depth X n_traj
# args.train_sampling_def = {'type': 'Generative', 'states_dist_from_uniform': 0.7, 'actions_dist_from_uniform': 0.7}  # n_samples =  args.depth X n_traj
# args.train_sampling_def = {'type': 'Generative', 'states_entropy': 1.5, 'actions_entropy': 1.0}  # n_samples =  args.depth X n_traj

# ----- Algorithm Parameters ---------------------------------------------#
args.initial_policy = 'uniform'  # 'uniform' (default) | 'generated_random'
args.epsilon = 0.1  # for epsilon-greedy
args.default_gamma = None  # default: None  # The default guidance discount factor (if None use gammaEval)
args.default_l2_factor = 1e-4  # default: None  # The default L2 factor (if using discount regularization) - note: it is necessary for LSTD
args.method = 'SARSA'  # 'Policy-Evaluation Algorithm'  # Options: 'Expected_SARSA'  | 'Model_Based' | 'SARSA' | 'LSTDQ' | 'ELSTDQ' | 'ELSTDQ_nested' | 'LSTDQ_nested'
args.use_reward_scaling = False

# Hyper-parameters for iterative methods ('SARSA', 'Expected_SARSA' )
args.TD_Init_type = 'zero'  # How to initialize V # Options: 'Vmax' | 'zero' | 'random_0_1' |  'random_0_Vmax' | '0.5_'Vmax' |
args.n_TD_iter = 5000    # Default: 500 for RandomMDP, 5000 for GridWorld  # number of TD iterations
args.learning_rate_def = {'type': 'a/(b+i_iter)', 'a': 500, 'b': 1000, 'scale': True}

# -------------------------------------------------------------------------------------------


# A Ray remote function.
# Runs a single repetition of the experiment
@ray.remote
def run_rep(i_rep, alg_param_grid, config_grid_def, args_r):

	set_random_seed(args_r.seed + i_rep)
	config_grid_vals = get_grid(config_grid_def)
	n_config_grid = len(config_grid_vals)
	n_grid = len(alg_param_grid)
	# runs a single repetition of the experiment
	loss_rep = np.zeros((n_config_grid, n_grid))
	gammaEval = args_r.gammaEval
	config_type = config_grid_def['type']

	# grid of number of trajectories to generate
	for i_config, config_val in enumerate(config_grid_vals):
		n_traj = args_r.n_trajectories
		if config_type == 'n_trajectories':
			n_traj = config_val
		elif config_type == 'states_actions_TV_dist_from_uniform':
			args_r.train_sampling_def = {'type': 'Generative', 'states_TV_dist_from_uniform': config_val, 'actions_TV_dist_from_uniform': config_val}
		elif config_type == 'chain_mix_time':
			args_r.train_sampling_def = {'type': 'chain_mix_time', 'mix_time': config_val}
		elif config_type == 'n_episodes':
			args_r.n_episodes = config_val
		elif config_type == 'None':
			pass
		else:
			raise AssertionError

		# Generate MDP:
		M = MDP(args_r)

		# Optimal policy for the MDP:
		pi_opt, V_opt, Q_opt = PolicyIteration(M, gammaEval)

		# grid of regularization param
		for i_grid, alg_param in enumerate(alg_param_grid):
				gamma_guidance, l1_factor, l2_factor = get_regularization_params(args_r, alg_param,
				                                                                 args_r.param_grid_def['type'])

				# run the learning episodes:
				pi_t = run_learning_method(args_r, M, n_traj, gamma_guidance, l2_factor, l1_factor)

				# Evaluate performance of learned policy:
				V_t, _ = PolicyEvaluation(M, pi_t, gammaEval)

				loss_rep[i_config, i_grid] = (np.abs(V_opt - V_t)).mean()
		# end for grid
	#  end for i_config
	return loss_rep


# end run_rep
# -------------------------------------------------------------------------------------------


def get_regularization_params(args_r, alg_param, reg_type):
	gammaEval = args_r.gammaEval
	if args_r.default_gamma is None:
		gamma_guidance = gammaEval
	else:
		gamma_guidance = args_r.default_gamma
	l2_factor = args.default_l2_factor
	l1_factor = None

	if reg_type in 'L2_factor':
		l2_factor = alg_param
	elif reg_type == 'L1_factor':
		l1_factor = alg_param
	elif reg_type == 'gamma_guidance':
		gamma_guidance = alg_param
	elif reg_type == 'NoReg':
		pass
	else:
		raise AssertionError('Unrecognized regularization type: ' + str(reg_type))
	return gamma_guidance, l1_factor, l2_factor


# -------------------------------------------------------------------------------------------


def run_simulations(args, save_result, local_mode, init_ray=True):
	if init_ray:
		start_ray(local_mode)
	if save_result:
		create_result_dir(args)
		write_to_log('local_mode == {}'.format(local_mode), args)

	start_time = timeit.default_timer()
	set_random_seed(args.seed)

	n_reps = args.n_reps
	alg_param_grid = get_grid(args.param_grid_def)
	n_grid = alg_param_grid.shape[0]
	config_grid_vals = get_grid(args.config_grid_def)
	n_config_grid = len(config_grid_vals)
	planing_loss = np.zeros((n_reps, n_config_grid, n_grid))
	info_dict = {}
	# ----- Run simulation in parrnell process---------------------------------------------#
	loss_rep_id_lst = []
	for i_rep in range(n_reps):
		# returns objects ids:
		args_r = deepcopy(args)
		planing_loss_rep_id = run_rep.remote(i_rep, alg_param_grid, args_r.config_grid_def, args_r)
		loss_rep_id_lst.append(planing_loss_rep_id)
	# end for i_rep
	# -----  get the results --------------------------------------------#
	for i_rep in range(n_reps):
		loss_rep = ray.get(loss_rep_id_lst[i_rep])
		if i_rep % max(n_reps // 100, 1) == 0:
			time_str = time.strftime("%H hours, %M minutes and %S seconds",
			                         time.gmtime(timeit.default_timer() - start_time))
			write_to_log('Finished: {} out of {} reps, time: {}'.format(i_rep + 1, n_reps, time_str), args)
		# end if
		planing_loss[i_rep] = loss_rep
		info_dict = {'planing_loss_avg': planing_loss.mean(axis=0), 'planing_loss_std': planing_loss.std(axis=0),
		             'alg_param_grid': alg_param_grid, 'n_reps_finished': i_rep + 1}
		if save_result:
			save_run_data(args, info_dict, verbose=0)
		# end if
	# end for i_rep
	if save_result:
		save_run_data(args, info_dict)
	stop_time = timeit.default_timer()
	write_to_log('Total runtime: ' +
	             time.strftime("%H hours, %M minutes and %S seconds", time.gmtime(stop_time - start_time)), args,
	             save_result)
	return info_dict


# end  run_simulations
# -------------------------------------------------------------------------------------------


def run_main_control(args, save_result=True, load_run_data_flag=False, result_dir_to_load='', save_PDF=False, plot=True,
                     local_mode=False, init_ray=True):
	SetMdpArgs(args)
	if load_run_data_flag:
		args, info_dict = load_run_data(result_dir_to_load)
	else:
		info_dict = run_simulations(args, save_result, local_mode, init_ray=init_ray)
	planing_loss_avg = info_dict['planing_loss_avg']
	planing_loss_std = info_dict['planing_loss_std']
	alg_param_grid = info_dict['alg_param_grid']
	if 'n_reps_finished' in info_dict.keys():
		n_reps_finished = info_dict['n_reps_finished']
	else:
		n_reps_finished = args.n_reps
	# end if
	# ----- Plot figures  ---------------------------------------------#

	if plot or save_PDF:
		ax = plt.figure().gca()
		if args.train_sampling_def['type'] in {'Generative', 'Generative_uniform', 'Generative_Stationary'}:
			data_size_per_traj = args.depth * args.n_episodes
		elif args.train_sampling_def['type'] == 'Trajectories':
			data_size_per_traj = args.depth * args.n_episodes
		elif args.train_sampling_def['type'] == 'sample_all_s_a':
			data_size_per_traj = args.nS * args.nA * args.n_episodes
		else:
			raise AssertionError
		# end if
		xscale = 1.
		legend_title = ''
		if args.param_grid_def['type'] == 'L2_factor':
			plt.xlabel(r'$L_2$ Regularization Factor [1e-2]')
			xscale = 1e2
		elif args.param_grid_def['type'] == 'L1_factor':
			plt.xlabel(r'$L_1$ Regularization Factor ')
		elif args.param_grid_def['type'] == 'gamma_guidance':
			plt.xlabel(r'Guidance Discount Factor $\gamma$')
		else:
			raise AssertionError('Unrecognized args.grid_type')
		# end if
		ci_factor = 1.96 / np.sqrt(n_reps_finished)  # 95% confidence interval factor

		config_grid_vals = get_grid(args.config_grid_def)
		for i_config, config_val in enumerate(config_grid_vals): # for of plots in the figure

			if args.config_grid_def['type'] == 'n_trajectories':
				if args.train_sampling_def['type'] in {'Generative_uniform', 'Generative', 'Generative_Stationary'}:
					legend_title = 'Num. Samples'
					label_str = '{} '.format(config_val * data_size_per_traj)
				else:
					legend_title = 'Num. Trajectories'
					label_str = '{} '.format(config_val)

			elif args.config_grid_def['type'] == 'states_actions_TV_dist_from_uniform':
				legend_title = 'Total-Variation from\n uniform [normalized]'
				label_str = '{} '.format(config_val)

			elif args.config_grid_def['type'] == 'chain_mix_time':
				legend_title = 'Mixing time'
				label_str = '{} '.format(config_val)

			elif args.config_grid_def['type'] == 'n_episodes':
				legend_title = 'Num. Episodes'
				label_str = '{} '.format(config_val)

			else:
				raise AssertionError
			# end if
			plt.errorbar(alg_param_grid * xscale, planing_loss_avg[i_config], yerr=planing_loss_std[i_config] * ci_factor,
			             marker='.', label=label_str)
			if show_stars:
				# Mark the lowest point:
				i_best = np.argmin(planing_loss_avg[i_config])
				plt.scatter(alg_param_grid[i_best] * xscale, planing_loss_avg[i_config][i_best], marker='*', s=400)
			# end if
		# for i_config
		plt.grid(True)
		plt.ylabel('Loss')
		plt.legend(title=legend_title, loc='best', fontsize=12)  # loc='upper right'
		if y_lim:
			plt.ylim(y_lim)
		# plt.xlim([0.5,1])
		# ax.set_yticks(np.arange(0., 9., step=1.))
		# plt.figure(figsize=(5.8, 3.0))  # set up figure size

		if save_PDF:
			save_fig(args.run_name)
		else:
			# plt.title('Loss +- 95% CI \n ' + str(args.args))
			plt.title(args.mdp_def['type'] + '  ' + args.run_name + ' \n ' + args.result_dir, fontsize=6)
		# end if save_PDF
	# end if
	pretty_print_args(args)
	if plot:
		plt.show()
	print('done')
	info_dict['result_dir'] = args.result_dir
	return info_dict


# end run_main


# -------------------------------------------------------------------------------------------
if __name__ == "__main__":
	info_dict = run_main_control(args, save_result=True, load_run_data_flag=load_run,
	                             result_dir_to_load=result_dir_to_load, save_PDF=save_PDF, local_mode=local_mode)

In [ ]:
# ===== File: main_Runs.py =====
# Uses code from:
# https://github.com/sfujim/TD3
# https://github.com/pranz24/pytorch-soft-actor-critic

import sys
import os
import argparse
import timeit
import time
import glob
import ray
import pickle
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
import ray
	create_results_backup, start_ray, save_fig, pretty_print_args

params = {'font.size': 10, 'lines.linewidth': 2, 'legend.fontsize': 10, 'legend.handlelength': 2, 'pdf.fonttype': 42,
		  'ps.fonttype': 42}
plt.rcParams.update(params)

project_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), '.'))
sys.path.insert(0, project_dir)

# --------------------------------------------------------------------------------------------------------------------#
#  Set parameters
# --------------------------------------------------------------------------------------------------------------------#

parser = argparse.ArgumentParser()
parser.add_argument("--alg", default="TD3")  # algorithm name 'TD3' 'SAC' |  'OurDDPG')
parser.add_argument("--env", default="Hopper-v2")  # OpenAI gym environment name
parser.add_argument("--seed", default=0, type=int)  # Sets Gym, PyTorch and Numpy seeds
parser.add_argument("--start_timesteps", default=1e4, type=int)  # Time steps initial random policy is used
parser.add_argument("--max_timesteps", default=2e5, type=int)  # Max time steps to run environment
parser.add_argument("--eval_freq", default=int(25e3), type=int)  # How often (time steps) we evaluate  and save 'snapshot'
parser.add_argument("--expl_noise", default=0.1)  # Std of Gaussian exploration noise
parser.add_argument("--batch_size", default=256, type=int)  # 256  # Batch size for both actor and critic
parser.add_argument("--default_discount", default=0.999)  # Default Discount factor
parser.add_argument("--tau", default=0.005)  # Target network update rate
parser.add_argument("--policy_noise", default=0.2)  # Noise added to target policy during critic update
parser.add_argument("--noise_clip", default=0.5)  # Range to clip target policy noise
parser.add_argument("--policy_freq", default=2, type=int)  # Frequency of delayed policy updates
parser.add_argument("--save_model", action="store_true")  # Save model and optimizer parameters
parser.add_argument("--load_model", default="")  # Model load file name, "" doesn't load, "default" uses file_name
parser.add_argument("--run_name",
					default="")  # ='Name of dir to save results  (of the grid search) in (if empty, name by time)',
parser.add_argument("--iter_per_sample", default=1)  #
args = parser.parse_args()

# hardware resources required for each worker:
num_gpus = 0.3  # how much of the GPU is required for each worker

# run config
smoke_test = False  # True/False - short run for debug
local_mode = False  # True/False - run non-parallel to get error messages and debugging

save_PDF = False  # False/True - save figures as PDF file
y_lim = None  #  None | [100, 900]
x_lim = None   #  None | [0.4, 1]


# Option to load previous run results or continue unfinished run or start a new run:
run_mode = 'New'  # 'New' / 'Load' / 'Continue' / 'ContinueNewGrid' / 'ContinueAddGrid' / 'LoadSnapshot'
increase_n_reps_in_loaded_grid = False  # True/False - if true and run_mode is a continue mode, then increase the number
# of reps im each point to be at least  args.n_reps  o.w, loaded grid point will stay with
# the number of reps they have


# If run_mode ==  'Load' / 'Continue' use this results dir:
result_dir_to_load = './saved/main_Runs/2020_06_14_17_32_16'

timesteps_snapshot_to_load = int(25e3)  # Used if run_mode=='LoadSnapshot'

args.n_reps = 20  # 100 # number of experiment repetitions for each point in grid
args.evaluation_num_episodes = 1000
args.save_model = False
#  how to create parameter grid:

args.param_grid_def = {'type': 'gamma_guidance', 'spacing': 'list', 'decimals': 4, 'list': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.985, 0.99, 0.995, 1.]}
# args.param_grid_def = {'type': 'gamma_guidance', 'spacing': 'linspace', 'start': 0.2, 'stop': 0.999, 'num': 10, 'decimals': 4}
# args.param_grid_def = {'type': 'L2_factor', 'spacing': 'linspace', 'start': 0.0, 'stop': 0.05, 'num': 11, 'decimals': 4}
# args.param_grid_def = {'type': 'L2_factor', 'spacing': 'list', 'list': [0.055, 0.06, 0.065, 0.07, 0.075, 0.08]}

args.default_critic_l2_reg = 0  # default L2 regularization factor for the Q-networks (critic)
args.actor_l2_reg = 0  # L2 regularization factor for the policy-networks (actor)

args.actor_hiddens = [400, 300]
args.critic_hiddens = [400, 300]

# ----------------------------------------------------------------------
if smoke_test:
	print('Smoke Test !!!!!!! \n' * 10)
	args.start_timesteps = 1e0
	args.max_timesteps = 1e1
	args.eval_freq = 2e0
	args.n_reps = 2
	args.evaluation_num_episodes = 2


# ----------------------------------------------------------------------
# define a remote function
@ray.remote(num_gpus=num_gpus, max_calls=1)
def run_simulation_remote(args_run, job_name, job_info):
	if args_run.alg in {'TD3', 'OurDDPG'}:
		from TD3_Code.mainTD3 import run_simulation_TD3
		write_to_log('Starting: {}, time: {}'.format(job_name, time_now()), args_run)
		args_run.run_name += ':' + job_name
		return run_simulation_TD3(args_run, job_info), args_run
	# elif  args_run.alg == 'SAC':
	# 	from SAC_Code.main_SAC import run_simulation_SAC
	# 	write_to_log('Starting: {}, time: {}'.format(job_name, time_now()), args_run)
	# 	args_run.run_name += ':' + job_name
	# 	return run_simulation_SAC(args_run, job_info), args_run
	else:
		raise AssertionError


# --------------------------------------------------------------------------------------------------------------------#

#  Get results
# --------------------------------------------------------------------------------------------------------------------#

# --------------------------------------------------------------------------------------------------------------------#

if run_mode in {'Load', 'Continue'}:
	print_status(result_dir_to_load)
	#  Load previous run
	args, info_dict = load_run_data(result_dir_to_load)
	args.result_dir = result_dir_to_load  # update the path, in case the result folder moved
	if 'result_reward_mat' not in info_dict.keys():
		raise AssertionError('run_mode=={}, but the path result_dir_to_load does not contain any finished results (try run_mode="LoadSnapshot")'.format(run_mode))
	alg_param_grid = info_dict['alg_param_grid']
	result_reward_mat = info_dict['result_reward_mat']
	run_time = info_dict['run_time']
	n_grid = len(alg_param_grid)
	n_reps_per_point = np.full(shape=n_grid, fill_value=args.n_reps)
	print('Loaded parameters: \n', args, '\n', '-' * 20)

# --------------------------------------------------------------------------------------------------------------------#

elif run_mode == 'LoadSnapshot':
	print_status(result_dir_to_load)
	#  Load previous run
	args, info_dict = load_run_data(result_dir_to_load)
	print('Loaded parameters: \n', args, '\n', '-' * 20)
	args.result_dir = result_dir_to_load  # update the path, in case the result folder moved
	alg_param_grid = get_grid(args.param_grid_def)
	run_time = info_dict['run_time']
	path = os.path.join(result_dir_to_load, 'jobs')
	all_files = glob.glob(os.path.join(path, "*.p"))
	n_grid = len(alg_param_grid)
	n_reps = args.n_reps
	n_reps_per_point = np.full(shape=n_grid, fill_value=0, dtype=np.int)
	result_reward_mat = np.full(shape=(n_grid, n_reps), fill_value=np.nan)

	for f_path in all_files:
		save_dict = pickle.load(open(f_path, "rb"))
		job_info = save_dict['job_info']
		timesteps_snapshots = save_dict['timesteps_snapshots']
		evaluations = save_dict['evaluations']
		load_idx = np.searchsorted(timesteps_snapshots, int(timesteps_snapshot_to_load), 'right')
		if load_idx == len(timesteps_snapshots):
			continue  # snapshot not found
		i_grid = np.searchsorted(alg_param_grid, job_info['grid_param'], 'right')
		if i_grid == len(alg_param_grid):
			# the loaded param is not in our defined alg_param_grid
			continue
			# TODO: add option to load all files, even if not in the defined alg_param_grid (show_all_saved_results flag)
		i_rep = job_info['i_rep']
		result_reward_mat[i_grid, n_reps_per_point[i_grid]] = evaluations[load_idx]
		n_reps_per_point[i_grid] += 1


# --------------------------------------------------------------------------------------------------------------------#

elif run_mode in {'ContinueNewGrid', 'ContinueAddGrid'}:
	print_status(result_dir_to_load)
	# Create a new gird according to param_grid_def defined above, and use the loaded results if compatible.
	# all the other run  args (besides param_grid_def) are according to the loaded file
	loaded_args, info_dict = load_run_data(result_dir_to_load)
	if 'alg' not in loaded_args: # legacy naming
		loaded_args.alg = loaded_args.policy
	assert loaded_args.param_grid_def['type'] == args.param_grid_def['type']
	create_results_backup(result_dir_to_load)
	loaded_alg_param_grid = info_dict['alg_param_grid']
	loaded_result_mat = info_dict['result_reward_mat']
	run_time = info_dict['run_time']
	args.result_dir = result_dir_to_load  # update the path, in case the result folder moved

	new_param_grid_def = args.param_grid_def
	desired_alg_param_grid = get_grid(new_param_grid_def)
	args = deepcopy(loaded_args)

	if run_mode == 'ContinueAddGrid':
		new_alg_param_grid = np.union1d(loaded_alg_param_grid, desired_alg_param_grid)
		new_alg_param_grid.sort()
		args.param_grid_def['spacing'] = 'list',
		args.param_grid_def['list'] = new_alg_param_grid

	else:  # run_mode == 'ContinueNewGrid'
		# in this case we omit loaded grid point that are not in desired_alg_param_grid
		args.param_grid_def = new_param_grid_def
		new_alg_param_grid = desired_alg_param_grid

	n_grid = len(new_alg_param_grid)
	desired_n_reps = args.n_reps

	# check how many completed results we have in loaded data:
	found_points = False
	finished_reps_per_point = np.zeros(n_grid, dtype=np.int)
	for i_grid, grid_param in enumerate(new_alg_param_grid):
		if grid_param in loaded_alg_param_grid:
			load_idx = np.nonzero(loaded_alg_param_grid == grid_param)
			found_points = True
			finished_reps_per_point[i_grid] = get_num_finished(loaded_result_mat[load_idx])
	# end for i_grid
	if not found_points:
		raise Warning('Loaded file  {} did not complete any of the desired grid points'.format(result_dir_to_load))

	# determine number of reps required in each grid point
	n_reps_per_point = np.zeros(n_grid, dtype=np.int)
	for i_grid, grid_param in enumerate(new_alg_param_grid):
		if grid_param in desired_alg_param_grid or increase_n_reps_in_loaded_grid:
			n_reps_per_point[i_grid] = max(desired_n_reps, finished_reps_per_point[i_grid])
		else:
			n_reps_per_point[i_grid] = finished_reps_per_point[i_grid]

	# now take completed results from loaded data:
	result_reward_mat = np.full((n_grid, np.max(n_reps_per_point)), np.nan)
	for i_grid, grid_param in enumerate(new_alg_param_grid):
		if grid_param in loaded_alg_param_grid:
			load_idx = np.nonzero(loaded_alg_param_grid == grid_param)
			for i_rep in range(finished_reps_per_point[i_grid]):
				result_reward_mat[i_grid, i_rep] = loaded_result_mat[load_idx, i_rep]
			# end for i_rep
		# end if
	# end for i_grid
	write_to_log('Continue run with new grid def {}, {}'.format(new_param_grid_def, time_now()), args)
	write_to_log('Run parameters: \n' + str(args) + '\n' + '-' * 20, args)
	pretty_print_args(args)
	alg_param_grid = new_alg_param_grid
# --------------------------------------------------------------------------------------------------------------------#

elif run_mode == 'New':
	# Start from scratch
	run_time = 0
	create_result_dir(args)
	os.makedirs(os.path.join(args.result_dir, 'jobs'))
	alg_param_grid = get_grid(args.param_grid_def)
	n_grid = len(alg_param_grid)
	n_reps = args.n_reps
	n_reps_per_point = np.full(shape=n_grid, fill_value=n_reps, dtype=np.int)
	result_reward_mat = np.full(shape=(n_grid, n_reps), fill_value=np.nan)
else:
	raise AssertionError('Unrecognized run_mode')
# --------------------------------------------------------------------------------------------------------------------#

if run_mode in {'New', 'Continue', 'ContinueNewGrid', 'ContinueAddGrid'}:

	# Run grid
	start_time = timeit.default_timer()
	start_ray(local_mode)
	write_to_log('Run grid == {}'.format(alg_param_grid), args)
	write_to_log('local_mode == {}'.format(local_mode), args)
	out_id = [[None for _ in range(n_reps_per_point[i_grid])] for i_grid in range(n_grid)]
	# check previously finished reps
	n_finished_g = np.zeros(n_grid, dtype=np.int)
	for i_grid, grid_param in enumerate(alg_param_grid):
		n_finished_g[i_grid] = get_num_finished(result_reward_mat[i_grid])
		write_to_log(
			f'Grid point {1 + i_grid}/{len(alg_param_grid)}, val: {grid_param},'
			f' Number of finished reps loaded: {n_finished_g[i_grid]}', args)
	# Run several repetitions of training:
	# note: the for i_rep is th outer loop so we will get initial results from all grid point, even if not accurate
	for i_rep in range(np.max(n_reps_per_point)):
		for i_grid, grid_param in enumerate(alg_param_grid):
			n_finished = n_finished_g[i_grid]  # number of reps already finished
			if n_finished <= i_rep < n_reps_per_point[i_grid]:
				args_run = deepcopy(args)
				# Set seed (unique for each repetition)
				args_run.seed = args.seed + i_rep
				# Set args with the grid value
				if args.param_grid_def['type'] == 'L2_factor':
					args_run.discount = args.default_discount
					args_run.critic_l2_reg = grid_param
					job_name = 'L2_' + str(grid_param)
				elif args.param_grid_def['type'] == 'gamma_guidance':
					args_run.discount = grid_param
					args_run.critic_l2_reg = args.default_critic_l2_reg
					job_name = 'Gamma_' + str(grid_param)
				else:
					raise AssertionError('Unrecognized args.grid_type')
				job_name += '_Rep_' + str(i_rep)
				args_run.job_name = job_name
				# Start the job and get the outputs objects id's:
				job_info = {'grid_param': grid_param, 'i_rep': i_rep}
				out_id[i_grid][i_rep] = run_simulation_remote.remote(args_run, job_name, job_info)
	# end for i_grid
	# end if
	# end for i_rep
	# collect the outputs of the finished jobs:
	for i_rep in range(np.max(n_reps_per_point)):
		for i_grid, grid_param in enumerate(alg_param_grid):
			# check if job is already finished (or not sent at all):
			if i_rep >= n_reps_per_point[i_grid] or not np.isnan(result_reward_mat[i_grid, i_rep]):
				continue  # skip
			output, args_run = ray.get(out_id[i_grid][i_rep])
			result_reward_mat[i_grid, i_rep] = output
			# note: the final reward is an average performance on eval_episodes=10 of final policy
			write_to_log(
				f'Finished Rep: {i_rep + 1}/{n_reps_per_point[i_grid]} of Grid point {i_grid}/{len(alg_param_grid)}'
				f' ({args_run.job_name}), Reward : {output}, Time now: {time_now()}', args)
			# Save results so far:
			stop_time = timeit.default_timer()
			run_time += stop_time - start_time
			start_time = timeit.default_timer()
			save_run_data(args, {'alg_param_grid': alg_param_grid, 'result_reward_mat': result_reward_mat,
								 'run_time': run_time}, verbose=1)
	# end for i_grid
	# end for i_rep

	write_to_log('Total runtime: ' + time.strftime("%H hours, %M minutes and %S seconds", time.gmtime(run_time)), args)

# --------------------------------------------------------------------------------------------------------------------#
#  Plot results
# --------------------------------------------------------------------------------------------------------------------#

mean_reward = []
std_reward = []
n_finished_g = []
grid_vals = []
for i_grid, val in enumerate(alg_param_grid):
	n_finished = get_num_finished(result_reward_mat[i_grid])
	if n_finished > 0:
		mean_r = np.mean(result_reward_mat[i_grid, :n_finished])
		mean_reward.append(mean_r)
		std_r = np.std(result_reward_mat[i_grid, :n_finished])
		std_reward.append(std_r)
		n_finished_g.append(n_finished)
		grid_vals.append(val)
		print(
			f'Grid point {i_grid + 1}/{len(alg_param_grid)}, Val {val}, #Reps Finished {n_finished},'
			f' Mean Reward {mean_r}, STD Reward {std_r}')

mean_reward = np.array(mean_reward)
std_reward = np.array(std_reward)
grid_vals = np.array(grid_vals)
n_finished_g = np.array(n_finished_g)
xscale = 1.
if args.param_grid_def['type'] == 'L2_factor':
	xscale = 1e2
	xlabel = r'$L_2$ Factor [1e-2]'
	title_prefix = args.env + r', $L_2$ Regularization'
elif args.param_grid_def['type'] == 'gamma_guidance':
	xlabel = r'Guidance Discount Factor $\gamma$'
	title_prefix = args.env + r', Discount Regularization'
else:
	raise AssertionError('Unrecognized args.grid_type')

if run_mode == 'LoadSnapshot':
	title_prefix += ', TimeSteps: {}'.format(int(timesteps_snapshot_to_load))
else:
	title_prefix += ', TimeSteps: {}'.format(int(args.max_timesteps))

# Plot number of finished reps
plt.figure()
plt.plot(xscale * grid_vals, n_finished_g, marker='o')
plt.grid(True)
plt.xlabel(xlabel)
plt.xlabel("# finished reps")

# Plot reward
ci_factor = 1.96 / np.sqrt(n_finished_g)  # 95% confidence interval factor
plt.figure()
plt.plot(xscale * grid_vals, mean_reward, marker='o')

plt.fill_between(xscale * grid_vals, mean_reward - std_reward * ci_factor, mean_reward + std_reward * ci_factor,
				 color='blue', alpha=0.2)
plt.grid(True)
plt.xlabel(xlabel)
if y_lim:
	plt.ylim(y_lim)
if x_lim:
	plt.xlim(x_lim)
plt.ylabel('Average Episode Return')
if save_PDF:
	plt.title(title_prefix)
	save_fig(args.run_name)
else:
	plt.title(f'{title_prefix}, reps_finished: {min(n_finished_g)} - {max(n_finished_g)}\n {args.result_dir}',
			  fontsize=8)
# + 'Episode Reward Mean +- 95% CI, ' + ' \n '
plt.show()

In [ ]:
# ===== File: run_2d_reg_grid_PolEval.py =====
"""
In  tabular MDP setting, evaluates the learning of optimal policy using different guidance discount factors
On-policy means we run episodes,
 in each episode we generate roll-outs/trajectories of current policy and run algorithm to improve the policy.
"""

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import argparse
from copy import deepcopy
import timeit
import time

from main_MRP import run_main_mrp

set_default_plot_params()


# -------------------------------------------------------------------------------------------
#  Run mode
# -------------------------------------------------------------------------------------------

load_run_data_flag = False  # False/True If true just load results from dir, o.w run simulation
result_dir_to_load = './saved/2020_02_04_06_19_35'  # '2020_02_04_06_19_35' | '2020_02_03_21_45_36'
save_PDF = False  # False/True - save figures as PDF file
local_mode = False  # True/False - run non-parallel to get error messages and debugging

# -------------------------------------------------------------------------------------------
#  Set Parameters
# -------------------------------------------------------------------------------------------

args = argparse.Namespace()

# ----- Run Parameters ---------------------------------------------#
args.run_name = ''   # 'Name of dir to save results in (if empty, name by time)'
args.seed = 1  # random seed
args.n_reps = 1000  # default 1000  # number of experiment repetitions

#  how to create parameter grid:
args.gam_grid_def = {'type': 'gamma_guidance', 'spacing': 'linspace', 'start': 0.9, 'stop': 0.99, 'num': 11, 'decimals': 10}
args.l2_grid_def = {'type': 'l2_factor', 'spacing': 'linspace', 'start': 0., 'stop': 0.1, 'num': 11, 'decimals': 10}

# ----- Problem Parameters ---------------------------------------------#

args.mrp_def = {'type': 'GridWorld', 'N0': 4, 'N1': 4, 'reward_std': 0.5, 'forward_prob_distrb': 'uniform', 'goal_reward': 1, 'R_low': -0.5, 'R_high': 0.5, 'policy': 'uniform'}

args.depth = 50  # default: 10 for 'chain', 100 for 'GridWorld'  # Length of trajectory
args.gammaEval = 0.99   # default: 0.99  # gammaEval

args.initial_state_distrb_type = 'uniform'  # 'uniform' | 'middle'
args.n_trajectories = 2  #

args.train_sampling_def = {'type': 'Trajectories'}
# args.train_sampling_def = {'type': 'Generative_uniform'}
# args.train_sampling_def = {'type': 'sample_all_s'}
args.config_grid_def = {'type': 'None', 'spacing': 'list', 'list': [None]}

args.evaluation_loss_type = 'L2_uni_weight' #  'rankings_kendalltau' | 'L2_uni_weight | 'L2' | 'one_pol_iter_l2_loss'


# ----- Algorithm Parameters ---------------------------------------------#
args.default_gamma = None  # default: None  # The default guidance discount factor (if None use gammaEval)

args.alg_type = 'LSTD'  # 'LSTD' | 'LSTD_Nested' | 'batch_TD_value_evaluation' | 'LSTD_Nested_Standard' | 'model_based_pol_eval' | 'model_based_known_P'
args.use_reward_scaling = False  # False | True.  set False for LSTD

args.base_lstd_l2_fp = 1e-5
args.base_lstd_l2_proj = 1e-4

# if batch_TD_value_evaluation is used:
args.default_l2_TD = None  # default: None  # The default L2 factor for TD (if using discount regularization)
args.TD_Init_type = 'zero'  # How to initialize V # Options: 'Vmax' | 'zero' | 'random_0_1' |  'random_0_Vmax' | '0.5_'Vmax' |
args.n_TD_iter = 5000  # Default: 500 for RandomMDP, 5000 for GridWorld  # number of TD iterations
args.learning_rate_def = {'type': 'a/(b+i_iter)', 'a': 500, 'b': 1000, 'scale': False}


# -------------------------------------------------------------------------------------------
def run_simulations(args, local_mode):
	start_ray(local_mode)
	create_result_dir(args)
	write_to_log('local_mode == {}'.format(local_mode), args)
	start_time = timeit.default_timer()
	create_result_dir(args)
	set_random_seed(args.seed)

	l2_grid = get_grid(args.l2_grid_def)
	gam_grid = get_grid(args.gam_grid_def)
	grid_shape = (len(l2_grid), len(gam_grid))
	loss_avg = np.zeros(grid_shape)
	loss_std = np.zeros(grid_shape)

	run_idx = 0
	for i0 in range(grid_shape[0]):
		for i1 in range(grid_shape[1]):
			args_run = deepcopy(args)
			args_run.param_grid_def = {'type': 'l2_factor', 'spacing': 'list', 'list': [l2_grid[i0]]}
			args_run.default_gamma = gam_grid[i1]

			info_dict = run_main_mrp(args_run, save_result=False, plot=False, local_mode=local_mode)
			loss_avg[i0, i1] = info_dict['loss_avg'][0]
			loss_std[i0, i1] = info_dict['loss_std'][0]
			run_idx += 1
			print("Finished {}/{}".format(run_idx, loss_avg.size))
		# end for
	# end for
	grid_results_dict = {'l2_grid': l2_grid, 'gam_grid': gam_grid, 'loss_avg': loss_avg,
						 'loss_std': loss_std}
	save_run_data(args, grid_results_dict)
	stop_time = timeit.default_timer()
	write_to_log('Total runtime: ' +
				 time.strftime("%H hours, %M minutes and %S seconds", time.gmtime(stop_time - start_time)), args)
	return grid_results_dict
# -------------------------------------------------------------------------------------------


if __name__ == "__main__":
	if load_run_data_flag:
		args, grid_results_dict = load_run_data(result_dir_to_load)
	else:
		grid_results_dict = run_simulations(args, local_mode)
	l2_grid = grid_results_dict['l2_grid']
	gam_grid = grid_results_dict['gam_grid']
	loss_avg = grid_results_dict['loss_avg']
	loss_std = grid_results_dict['loss_std']

	ci_factor = 1.96 / np.sqrt(args.n_reps)  # 95% confidence interval factor
	max_deviate = 100. * np.max(loss_std * ci_factor / loss_avg)
	print('Max 95% CI relative to mean: ', max_deviate, '%')

	with sns.axes_style("white"):
		ax = sns.heatmap(loss_avg,  cmap="YlGnBu", xticklabels=gam_grid, yticklabels=l2_grid * 1e3,  annot=True)
		plt.xlabel(r'Guidance Discount Factor $\gamma$')
		plt.ylabel(r'$L_2$ Regularization Factor [1e-3]')
		if save_PDF:
			save_fig(args.run_name)
		else:
			plt.title('Loss avg. Max 95% CI relative to mean: {}%\n {}'.format(np.around(max_deviate, decimals=1), args.run_name))
		plt.show()
		print('done')

In [ ]:
# ===== File: run_2d_reg_grid_PolOpt.py =====
"""
In  tabular MDP setting, evaluates the learning of optimal policy using different guidance discount factors
On-policy means we run episodes,
 in each episode we generate roll-outs/trajectories of current policy and run algorithm to improve the policy.
"""

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import argparse
from copy import deepcopy
import timeit
import time

from main_control import run_main_control

set_default_plot_params()


# -------------------------------------------------------------------------------------------
#  Run mode
# -------------------------------------------------------------------------------------------

load_run_data_flag = False  # False/True If true just load results from dir, o.w run simulation
result_dir_to_load = './saved/run_2d_reg_grid_PolOpt/2020_06_28_11_59_58'  #
save_PDF = False  # False/True - save figures as PDF file
local_mode = False  # True/False - run non-parallel to get error messages and debugging

# -------------------------------------------------------------------------------------------
#  Set Parameters
# -------------------------------------------------------------------------------------------

args = argparse.Namespace()

# ----- Run Parameters ---------------------------------------------#
args.run_name = ''   # 'Name of dir to save results in (if empty, name by time)'
args.seed = 1  # random seed
args.n_reps = 1000  # default 1000  # number of experiment repetitions

#  how to create parameter grid:
args.l2_grid_def =  {'type': 'L2_factor', 'spacing': 'linspace', 'start': 0, 'stop': 0.005, 'num': 11, 'decimals': 10}
args.gam_grid_def = {'type': 'gamma_guidance', 'spacing': 'linspace', 'start': 0.79, 'stop': 0.99, 'num': 11, 'decimals': 10}

# ----- Problem Parameters ---------------------------------------------#

# ----- Problem Parameters ---------------------------------------------#
# MDP definition ( see data_utils.SetMdpArgs)
# args.mdp_def = {'type': 'RandomMDP', 'S': 10, 'A': 5, 'k': 2, 'reward_std': 0.1}
args.mdp_def = {'type': 'GridWorld', 'N0': 4, 'N1': 4, 'reward_std': 0.1, 'forward_prob_distrb': 'uniform',  'goal_reward': 1, 'R_low': -0.5, 'R_high': 0.5}
# args.mdp_def = {'type': 'GridWorld', 'N0': 4, 'N1': 4, 'reward_std': 0.1, 'forward_prob_distrb': {'alpha': 3, 'beta': 1}, 'goal_reward': 1}


args.depth = 10  # default: 10  # Length of trajectory
args.gammaEval = 0.99   # default: 0.99  # gammaEval
args.n_episodes = 5  # Number of episodes
args.n_trajectories = 16   # default number of trajectories to generate per episode
args.train_sampling_def = {'type': 'Trajectories'}
args.config_grid_def = {'type': 'None', 'spacing': 'list', 'list': [None]}

# ----- Algorithm Parameters ---------------------------------------------#
args.method = 'SARSA'  # default: 'Expected_SARSA' # 'RL Algorithm'  # Options: 'Model_Based' | 'SARSA' | Expected_SARSA
args.TD_Init_type = 'zero'   # How to initialize V # Options: 'Vmax' (default) | 'zero' | 'random_0_1' |  'random_0_'Vmax'' | '0.5_'Vmax' |
args.use_reward_scaling = True
args.n_TD_iter = 5000  # Default: 500 for RandomMDP, 5000 for GridWorld  # number of TD iterations
args.epsilon = 0.1  # for epsilon-greedy
args.learning_rate_def = {'type': 'a/(b+i_iter)', 'a': 500, 'b': 1000, 'scale': True}
args.default_gamma = None  # default: None  # The default guidance discount factor (if None use gammaEval)
args.default_l2_factor = 1e-4    # default: None  # The default L2 factor (if using discount regularization) - note: it is necessary for LSTD

# -------------------------------------------------------------------------------------------
def run_simulations(args, local_mode):
	start_ray(local_mode)
	create_result_dir(args)
	write_to_log('local_mode == {}'.format(local_mode), args)
	start_time = timeit.default_timer()
	create_result_dir(args)
	set_random_seed(args.seed)

	l2_grid = get_grid(args.l2_grid_def)
	gam_grid = get_grid(args.gam_grid_def)
	write_to_log('gamma_grid == {}'.format(gam_grid), args)
	write_to_log('l2_grid == {}'.format(l2_grid), args)
	grid_shape = (len(l2_grid), len(gam_grid))
	loss_avg = np.zeros(grid_shape)
	loss_std = np.zeros(grid_shape)

	run_idx = 0
	for i0 in range(grid_shape[0]):
		for i1 in range(grid_shape[1]):
			args_run = deepcopy(args)
			args_run.param_grid_def = {'type': 'L2_factor', 'spacing': 'list', 'list': [l2_grid[i0]]}
			args_run.default_gamma = gam_grid[i1]

			info_dict = run_main_control(args_run, save_result=False, plot=False, init_ray=False)
			loss_avg[i0, i1] = info_dict['planing_loss_avg'][0]
			loss_std[i0, i1] = info_dict['planing_loss_std'][0]
			run_idx += 1
			print("Finished {}/{}".format(run_idx, loss_avg.size))
		# end for
	# end for
	grid_results_dict = {'l2_grid': l2_grid, 'gam_grid': gam_grid, 'loss_avg': loss_avg,
						 'loss_std': loss_std}
	save_run_data(args, grid_results_dict)
	stop_time = timeit.default_timer()
	write_to_log('Total runtime: ' +
				 time.strftime("%H hours, %M minutes and %S seconds", time.gmtime(stop_time - start_time)), args)
	return grid_results_dict
# -------------------------------------------------------------------------------------------


if __name__ == "__main__":
	if load_run_data_flag:
		args, grid_results_dict = load_run_data(result_dir_to_load)
	else:
		grid_results_dict = run_simulations(args, local_mode)
	l2_grid = grid_results_dict['l2_grid']
	gam_grid = grid_results_dict['gam_grid']
	loss_avg = grid_results_dict['loss_avg']
	loss_std = grid_results_dict['loss_std']

	ci_factor = 1.96 / np.sqrt(args.n_reps)  # 95% confidence interval factor
	max_deviate = 100. * np.max(loss_std * ci_factor /  loss_avg)
	print('Max 95% CI relative to mean: ', max_deviate, '%')

	# fig, ax = plt.subplots(figsize=(7, 7))
	with sns.axes_style("white"):

		yticklabels = np.around(l2_grid * 1e5, decimals=3)
		yticklabels = np.round(yticklabels).astype(int)
		xticklabels = np.around(gam_grid, decimals=3)
		ax = sns.heatmap(loss_avg,  cmap="YlGnBu", xticklabels=xticklabels, yticklabels = yticklabels,  annot=True, annot_kws={"size": 8})
		ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
		plt.xlabel(r'Guidance Discount Factor $\gamma$')
		plt.ylabel(r'$L_2$ Regularization Factor [1e-5')
		if save_PDF:
			save_fig(args.run_name)
		else:
			plt.title('Loss avg. Max 95% CI relative to mean: {}%\n {}'.format(np.around(max_deviate, decimals=1), args.run_name))
		plt.show()
		print('done')

In [ ]:
# ===== File: run_hyper_grid_MRP.py =====
######## Code structure ######
# --- simulations
# for i_rep
# for i_hyper_grid
# for reg_type
# for i_pram_grid
# ray put
# for i_pram_grid
# ray get

# print finished i_rep/n_reps
# --- results processing
# for i_hyper_grid
# for reg_type
# average the n_reps results (or how much finished)
# find best reg param
# record best_loss[reg_type][i_hyper_grid]
# record best_param[reg_type][i_hyper_grid]
########
########

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import argparse
import timeit
import time
import ray
from copy import deepcopy
from collections import OrderedDict

	get_grid, start_ray, set_default_plot_params, save_fig


set_default_plot_params()

# -------------------------------------------------------------------------------------------
#  Run mode
# -------------------------------------------------------------------------------------------
run_mode = 'New'   #  'New'  / 'Load' / 'Continue'
result_dir_to_load = './saved/run_hyper_grid/2020_05_17_14_52_29'
save_PDF = False  # False/True - save figures as PDF file/

local_mode = False  # True/False - run non-parallel to get error messages and debugging

# -------------------------------------------------------------------------------------------
#  Set Parameters
# -------------------------------------------------------------------------------------------

args = argparse.Namespace()

# -----  Run Parameters ---------------------------------------------#
args.run_name = ''  # 'Name of dir to save results in (if empty, name by time)'
args.seed = 1  # random seed
args.n_reps = 10  # default 1000  # number of experiment repetitions

args.reg_types = ['gamma_guidance', 'l2_factor', 'NoReg']

args.evaluation_loss_type = 'L2_uni_weight' #  'rankings_kendalltau' | 'L2_uni_weight | 'L2' | 'one_pol_iter_l2_loss'


# -----  Hyper-grid definition ---------------------------------------------#

# args.hyper_grid_def = {'type': 'states_entropy', 'spacing': 'linspace', 'start': 0.4, 'stop': 1.0, 'num': 11,
#                             'decimals': 5} # note: below 0.4 is very slow

args.hyper_grid_def = {'type': 'states_TV_dist_from_uniform', 'spacing': 'linspace', 'start': 0.0, 'stop': 0.8,
                       'num': 9, 'decimals': 5}  # note: above 0.9 is very slow

# args.hyper_grid_def = {'type': 'Exploration_epsilon', 'spacing': 'linspace', 'start': 0.0, 'stop': 0.9, 'num': 9,
#                             'decimals': 5}

# args.hyper_grid_def = {'type': 'Connectivity_k', 'spacing': 'list', 'list': list(np.arange(1, 6, dtype=np.int))}


# ----- Search grid definition ---------------------------------------------#
args.search_grid_def = dict()

args.search_grid_def['gamma_guidance'] = {'type': 'gamma_guidance', 'spacing': 'linspace',
                                          'start': 0.7, 'stop': 0.99, 'num': 100, 'decimals': 10}

args.search_grid_def['l2_factor'] = {'type': 'l2_factor', 'spacing': 'linspace',
                                     'start': 0, 'stop': 1.2, 'num': 100, 'decimals': 10}

args.search_grid_def['NoReg'] = {'type': 'NoReg', 'spacing': 'list', 'list': [None]}

# ----- Problem Parameters ---------------------------------------------#
args.depth = 50  # default: 10  # Length of trajectory
args.gammaEval = 0.99  # default: 0.99  # gammaEval
args.n_trajectories = 2  # number of trajectories to generate per episode

# args.mrp_def = {'type': 'ToyMRP', 'p01': 0.5, 'p10': 0.5,  'reward_std': 0.1}
# args.mrp_def = {'type': 'Chain', 'p_left': 0.5, 'length': 9,  'mean_reward_range': (0, 1), 'reward_std': 0.1}
# args.mrp_def = {'type': 'RandomMDP', 'S': 100, 'A': 2, 'k': 5, 'reward_std': 0.1, 'policy': 'uniform'}
args.mrp_def = {'type': 'GridWorld', 'N0': 4, 'N1': 4, 'reward_std': 0.5, 'forward_prob_distrb': 'uniform', 'goal_reward': 1, 'R_low': -0.5, 'R_high': 0.5, 'policy': 'uniform'}

args.initial_state_distrb_type = 'uniform'  # 'uniform' | 'middle'

# ----- Algorithm Parameters ---------------------------------------------#
args.default_gamma = None  # default: None  # The default guidance discount factor (if None use gammaEval)

args.alg_type = 'batch_TD_value_evaluation'  # 'LSTD' | 'LSTD_Nested' | 'batch_TD_value_evaluation' | 'LSTD_Nested_Standard' | 'model_based_pol_eval' | 'model_based_known_P'
args.use_reward_scaling = False  # False | True.  set False for LSTD

# args.base_lstd_l2_fp = 1e-5
args.base_lstd_l2_proj = 1e-4

# if batch_TD_value_evaluation is used:
args.default_l2_TD = None  # default: None  # The default L2 factor for TD (if using discount regularization)
args.TD_Init_type = 'zero'  # How to initialize V # Options: 'Vmax' | 'zero' | 'random_0_1' |  'random_0_Vmax' | '0.5_'Vmax' |
args.n_TD_iter = 5000  # Default: 500 for RandomMDP, 5000 for GridWorld  # number of TD iterations
args.learning_rate_def = {'type': 'a/(b+i_iter)', 'a': 500, 'b': 1000, 'scale': False}


# -------------------------------------------------------------------------------------------
def set_hyper_param(args_r, hyper_grid_val):
	hyper_grid_type = args.hyper_grid_def['type']

	if hyper_grid_type == 'states_entropy':
		assert 'train_sampling_def' not in args_r
		args_r.train_sampling_def = {'type': 'Generative', 'states_entropy': hyper_grid_val}

	elif hyper_grid_type == 'states_TV_dist_from_uniform':
		assert 'train_sampling_def' not in args_r
		args_r.train_sampling_def = {'type': 'Generative', 'states_TV_dist_from_uniform': hyper_grid_val}

	else:
		raise AssertionError("Invalid args.hyper_grid_def['type']")
# -------------------------------------------------------------------------------------------


def get_regularization_params(args_r, reg_param, reg_type):
	gammaEval = args_r.gammaEval
	if args_r.default_gamma is None:
		gamma_guidance = gammaEval
	else:
		gamma_guidance = args_r.default_gamma

	alg_type = args_r.alg_type
	l2_proj = 0
	l2_fp = 0
	l2_TD = 0
	if alg_type in {'LSTD'}:
		l2_proj = args_r.base_lstd_l2_proj
	elif alg_type in {'LSTD_Nested', 'LSTD_Nested_Standard'}:
		l2_fp = args_r.base_lstd_l2_fp
		l2_proj = args_r.base_lstd_l2_proj
	elif alg_type in {'batch_TD_value_evaluation'}:
		l2_TD = args_r.default_l2_TD
	else:
		raise AssertionError

	if reg_type == 'gamma_guidance':
		gamma_guidance = reg_param

	elif reg_type == 'l2_factor':
		l2_proj += reg_param
		l2_TD = reg_param
		l2_fp += reg_param

	elif reg_type == 'NoReg':
		pass

	else:
		raise AssertionError
	return gamma_guidance, l2_TD, l2_fp, l2_proj
# -------------------------------------------------------------------------------------------


# A Ray remote function.
# Runs a single  experiment
@ray.remote
def run_exp(i_rep, args_run, reg_type, reg_param):
	# set seed
	set_random_seed(args_run.seed + i_rep)

	# Generate MDP and sampling distribution (with specified uniformity)
	M = MRP(args_run)

	gammaEval = args_run.gammaEval

	# set regularisation parameters
	gamma_guidance, l2_TD, l2_fp, l2_proj = get_regularization_params(args_run, reg_param, reg_type)

	# Generate data:
	data = M.SampleDataMrp(args_run)

	V_est, V_true = run_value_estimation_method(data, M, args_run, gamma_guidance, l2_proj, l2_fp, l2_TD)

	loss_type = args_run.evaluation_loss_type
	pi = None
	eval_loss = evaluate_value_estimation(loss_type, V_true, V_est, M, pi, gammaEval, gamma_guidance)

	return eval_loss

# -------------------------------------------------------------------------------------------

def run_simulation(args, hyper_grid_vals, loss, reg_grids, local_mode):
	start_ray(local_mode)
	write_to_log('local_mode == {}'.format(local_mode), args)
	SetMrpArgs(args)
	start_time = timeit.default_timer()
	set_random_seed(args.seed)

	reg_types = args.reg_types

	n_hyper_grid = len(hyper_grid_vals)
	n_reps = args.n_reps
	results_dict = dict()

	write_to_log('***** Starting  {} reps'.format(n_reps), args)
	for i_rep in range(n_reps):

		for i_hyper_grid, hyper_grid_val in enumerate(hyper_grid_vals):
			args_run = deepcopy(args)
			set_hyper_param(args_run, hyper_grid_val)

			# send jobs:
			out_ids = {reg_type: [None for _ in range(len(reg_grids[reg_type]))] for reg_type in reg_types}
			for reg_type in reg_types:

				for i_reg_pram, reg_param in enumerate(reg_grids[reg_type]):
					# ray put
					if np.isnan(loss[reg_type][i_hyper_grid, i_reg_pram, i_rep]):
						out_ids[reg_type][i_reg_pram] = run_exp.remote(i_rep, args_run, reg_type, reg_param)
				# end if
			# end for i_reg_pram
			# end for reg_type

			# Gather results:
			for reg_type in reg_types:
				for i_reg_pram, reg_param in enumerate(reg_grids[reg_type]):
					# ray get
					if out_ids[reg_type][i_reg_pram] is not None:
						out = ray.get(out_ids[reg_type][i_reg_pram])
						loss[reg_type][i_hyper_grid, i_reg_pram, i_rep] = out
				# end if
			# end for i_reg_pram
		# end for reg_type
		# end for i_hyper_grid

		# Save results so far
		results_dict = {'hyper_grid_vals': hyper_grid_vals, 'loss': loss, 'reg_grids': reg_grids, 'n_reps_finished': i_rep + 1}
		save_run_data(args, results_dict, verbose=0)
		time_str = time.strftime("%H hours, %M minutes and %S seconds",  time.gmtime(timeit.default_timer() - start_time))
		write_to_log('Finished: {} out of {} reps, time: {}'.format(i_rep + 1, n_reps, time_str), args)

	# end for i_rep

	stop_time = timeit.default_timer()
	write_to_log('Total runtime: ' + time.strftime("%H hours, %M minutes and %S seconds", time.gmtime(stop_time - start_time)), args)
	return results_dict


# end  run_simulations

# -------------------------------------------------------------------------------------------

if __name__ == "__main__":
	# *********************************
	if run_mode == 'Load':
		args, results_dict = load_run_data(result_dir_to_load)
	# *********************************
	elif run_mode == 'New':

		hyper_grid_vals = get_grid(args.hyper_grid_def)
		create_result_dir(args)
		n_hyper_grid = len(hyper_grid_vals)
		n_reps = args.n_reps
		reg_types = args.reg_types

		# define search grids for regularization parameters
		reg_grids = dict()
		for reg_type in reg_types:
			reg_param_grid_def = args.search_grid_def[reg_type]
			reg_grids[reg_type] = get_grid(reg_param_grid_def)

		# init result matrix with nan (no result)
		loss = {reg_type: np.full((n_hyper_grid, len(reg_grids[reg_type]), n_reps), np.nan) for reg_type in reg_types}

		# run
		results_dict = run_simulation(args, hyper_grid_vals, loss, reg_grids, local_mode)
		save_run_data(args, results_dict)
	# *********************************
	elif run_mode == 'Continue':

		loaded_args, loaded_results_dict = load_run_data(result_dir_to_load)
		args = loaded_args
		args.result_dir = result_dir_to_load  # update the path, in case the result folder moved
		hyper_grid_vals = loaded_results_dict['hyper_grid_vals']
		loss = loaded_results_dict['loss']
		reg_grids = loaded_results_dict['reg_grids']
		results_dict = run_simulation(args, hyper_grid_vals, loss, reg_grids, local_mode)
		save_run_data(args, results_dict)
	# *********************************
	else:
		raise AssertionError('Unrecognized run_mode')
	# *********************************

	hyper_grid_vals = results_dict['hyper_grid_vals']
	loss = results_dict['loss']
	reg_grids = results_dict['reg_grids']
	n_reps_finished = results_dict['n_reps_finished']
	n_hyper_grid = len(hyper_grid_vals)
	reg_types = args.reg_types

	# get results statistics
	best_reg_param = {reg_type: np.full((n_hyper_grid), np.nan) for reg_type in reg_types}
	best_loss_mean = {reg_type: np.full((n_hyper_grid), np.nan) for reg_type in reg_types}
	best_loss_std = {reg_type: np.full((n_hyper_grid), np.nan) for reg_type in reg_types}

	# get results statistics
	for reg_type in reg_types:

		for i_hyper_grid, hyper_grid_val in enumerate(hyper_grid_vals):
			# average results over reps:
			loss_curr = loss[reg_type][i_hyper_grid, :, :n_reps_finished]
			loss_m = np.mean(loss_curr, axis=1)

			# Mark the best reg pram for current uniformity val:
			i_best = loss_m.argmin(axis=0)

			best_reg_param[reg_type][i_hyper_grid] = reg_grids[reg_type][i_best]
			best_loss_mean[reg_type][i_hyper_grid] = loss_curr[i_best].mean()
			best_loss_std[reg_type][i_hyper_grid] = loss_curr[i_best].std()

	# end or i_hyper_grid
	# end for reg_type

	SetMrpArgs(args)

	reg_labels = OrderedDict([('NoReg', 'No Regularization'),
	                          ('gamma_guidance', 'Best Discount Regularization'),
	                          ('l2_factor', 'Best L2 Regularization')])

	grid_type = args.hyper_grid_def['type']
	xscale = 1.
	is_int_grid = False
	title_prefix = args.mdp_def['type'] + ' - ' + grid_type
	grid_label = 'Hyper-parameter'
	if grid_type in {'states_entropy', 'states_actions_entropy'}:
		grid_label = r'Entropy [normalized]'
	elif grid_type in {'states_TV_dist_from_uniform', 'states_actions_TV_dist_from_uniform'}:
		grid_label = r'Total-Variation from uniform [normalized]'
	elif grid_type == 'Exploration_epsilon':
		grid_label = r'Exploration parameter $epsilon$'
	elif grid_type == 'Connectivity_k':
		grid_label = r'Connectivity parameter $k$'
		hyper_grid_vals = [int(k) for k in hyper_grid_vals]
		is_int_grid = True

	# plot best reg params
	fig, ax = plt.subplots(2, 1)
	ax[0].errorbar(xscale * hyper_grid_vals, best_reg_param['gamma_guidance'])
	ax[0].grid(True)
	ax[0].set_xlabel(grid_label, fontsize=10)
	ax[0].set_ylabel('Best discount parameter', fontsize=10)
	ax[1].errorbar(xscale * hyper_grid_vals, 1e3 * best_reg_param['l2_factor'])
	ax[1].grid(True)
	ax[1].set_xlabel(grid_label, fontsize=10)
	ax[1].set_ylabel('Best L2 factor [1e3]', fontsize=10)
	if is_int_grid:
		ax.xaxis.set_major_locator(MaxNLocator(integer=True))

	# plot loss
	ax = plt.figure().gca()
	ci_factor = 1.96 / np.sqrt(n_reps_finished)  # 95% confidence interval factor
	for reg_type in reg_labels.keys():
		plt.errorbar(xscale * hyper_grid_vals, best_loss_mean[reg_type], best_loss_std[reg_type] * ci_factor, label=reg_labels[reg_type])
	plt.grid(True)
	plt.legend(fontsize=13)
	plt.xlabel(grid_label)
	plt.ylabel('Loss')
	if is_int_grid:
		ax.xaxis.set_major_locator(MaxNLocator(integer=True))

	if save_PDF:
		save_fig(args.run_name, base_path=args.result_dir)
		#
		# # plot loss gain
		# ax = plt.figure().gca()
		# ci_factor = 1.96 / np.sqrt(n_reps_finished)  # 95% confidence interval factor
		# for reg_type in reg_labels.keys():
		# 	if reg_type != 'NoReg':
		# 		plt.errorbar(xscale * hyper_grid_vals, best_loss_mean['NoReg'] - best_loss_mean[reg_type], (best_loss_std[reg_type] + best_loss_std['NoReg'])* ci_factor,
		# 					 label=reg_labels[reg_type])
		# 		# TODO:  more accurate  std
		# plt.grid(True)
		# plt.legend()
		# plt.xlabel(grid_label)
		# plt.ylabel('Loss Gain')
		# if  is_int_grid:
		# 	ax.xaxis.set_major_locator(MaxNLocator(integer=True))
		# if save_PDF:
		# 	save_fig(args.run_name)
	else:
		plt.title(title_prefix + ' \n ' + args.result_dir + ', reps_finished: ' + str(n_reps_finished), fontsize=8)
	# + 'Episode Reward Mean +- 95% CI, ' + ' \n '

	plt.show()
	print('done')

In [ ]:
# ===== File: run_hyper_grid_PolOpt.py =====
"""
In  tabular MDP setting, evaluates the learning of optimal policy using different guidance discount factors
On-policy means we run episodes,
 in each episode we generate roll-outs/trajectories of current policy and run algorithm to improve the policy.
"""

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import argparse
from copy import deepcopy
import timeit
import time

from main_control import run_main_control

set_default_plot_params()


# -------------------------------------------------------------------------------------------
#  Run mode
# -------------------------------------------------------------------------------------------

load_run_data_flag = False  # False/True If true just load results from dir, o.w run simulation
result_dir_to_load = './saved/run_2d_reg_grid_PolOpt/2020_06_30_09_14_43/'  #
save_PDF = False  # False/True - save figures as PDF file
local_mode = False  # True/False - run non-parallel to get error messages and debugging

# -------------------------------------------------------------------------------------------
#  Set Parameters
# -------------------------------------------------------------------------------------------

args = argparse.Namespace()

# ----- Run Parameters ---------------------------------------------#
args.run_name = ''   # 'Name of dir to save results in (if empty, name by time)'
args.seed = 1  # random seed
args.n_reps = 1000  # default 1000  # number of experiment repetitions

#  how to create parameter grid:
args.l2_grid_def =  {'type': 'L2_factor', 'spacing': 'linspace', 'start': 0, 'stop': 0.005, 'num': 11, 'decimals': 10}
args.gam_grid_def = {'type': 'gamma_guidance', 'spacing': 'linspace', 'start': 0.79, 'stop': 0.99, 'num': 11, 'decimals': 10}

# ----- Problem Parameters ---------------------------------------------#

# ----- Problem Parameters ---------------------------------------------#
# MDP definition ( see data_utils.SetMdpArgs)
# args.mdp_def = {'type': 'RandomMDP', 'S': 10, 'A': 5, 'k': 2, 'reward_std': 0.1}
args.mdp_def = {'type': 'GridWorld', 'N0': 4, 'N1': 4, 'reward_std': 0.1, 'forward_prob_distrb': 'uniform',  'goal_reward': 1, 'R_low': -0.5, 'R_high': 0.5}
# args.mdp_def = {'type': 'GridWorld', 'N0': 4, 'N1': 4, 'reward_std': 0.1, 'forward_prob_distrb': {'alpha': 3, 'beta': 1}, 'goal_reward': 1}


args.depth = 10  # default: 10  # Length of trajectory
args.gammaEval = 0.99   # default: 0.99  # gammaEval
args.n_episodes = 5  # Number of episodes
args.n_trajectories = 8   # default number of trajectories to generate per episode
args.train_sampling_def = {'type': 'Trajectories'}
args.config_grid_def = {'type': 'None', 'spacing': 'list', 'list': [None]}

# ----- Algorithm Parameters ---------------------------------------------#
args.method = 'SARSA'  # default: 'Expected_SARSA' # 'RL Algorithm'  # Options: 'Model_Based' | 'SARSA' | Expected_SARSA
args.TD_Init_type = 'zero'   # How to initialize V # Options: 'Vmax' (default) | 'zero' | 'random_0_1' |  'random_0_'Vmax'' | '0.5_'Vmax' |
args.use_reward_scaling = False
args.n_TD_iter = 5000  # Default: 500 for RandomMDP, 5000 for GridWorld  # number of TD iterations
args.epsilon = 0.1  # for epsilon-greedy
args.learning_rate_def = {'type': 'a/(b+i_iter)', 'a': 500, 'b': 1000, 'scale': True}
args.default_gamma = None  # default: None  # The default guidance discount factor (if None use gammaEval)
args.default_l2_factor = 1e-4    # default: None  # The default L2 factor (if using discount regularization) - note: it is necessary for LSTD

# -------------------------------------------------------------------------------------------
def run_simulations(args, local_mode):
	start_ray(local_mode)
	create_result_dir(args)
	write_to_log('local_mode == {}'.format(local_mode), args)
	start_time = timeit.default_timer()
	create_result_dir(args)
	set_random_seed(args.seed)

	l2_grid = get_grid(args.l2_grid_def)
	gam_grid = get_grid(args.gam_grid_def)
	write_to_log('gamma_grid == {}'.format(gam_grid), args)
	write_to_log('l2_grid == {}'.format(l2_grid), args)
	grid_shape = (len(l2_grid), len(gam_grid))
	loss_avg = np.zeros(grid_shape)
	loss_std = np.zeros(grid_shape)

	run_idx = 0
	for i0 in range(grid_shape[0]):
		for i1 in range(grid_shape[1]):
			args_run = deepcopy(args)
			args_run.param_grid_def = {'type': 'L2_factor', 'spacing': 'list', 'list': [l2_grid[i0]]}
			args_run.default_gamma = gam_grid[i1]

			info_dict = run_main_control(args_run, save_result=False, plot=False, init_ray=False)
			loss_avg[i0, i1] = info_dict['planing_loss_avg'][0]
			loss_std[i0, i1] = info_dict['planing_loss_std'][0]
			run_idx += 1
			print("Finished {}/{}".format(run_idx, loss_avg.size))
		# end for
	# end for
	grid_results_dict = {'l2_grid': l2_grid, 'gam_grid': gam_grid, 'loss_avg': loss_avg,
						 'loss_std': loss_std}
	save_run_data(args, grid_results_dict)
	stop_time = timeit.default_timer()
	write_to_log('Total runtime: ' +
				 time.strftime("%H hours, %M minutes and %S seconds", time.gmtime(stop_time - start_time)), args)
	return grid_results_dict
# -------------------------------------------------------------------------------------------


if __name__ == "__main__":
	if load_run_data_flag:
		args, grid_results_dict = load_run_data(result_dir_to_load)
	else:
		grid_results_dict = run_simulations(args, local_mode)
	l2_grid = grid_results_dict['l2_grid']
	gam_grid = grid_results_dict['gam_grid']
	loss_avg = grid_results_dict['loss_avg']
	loss_std = grid_results_dict['loss_std']

	ci_factor = 1.96 / np.sqrt(args.n_reps)  # 95% confidence interval factor
	max_deviate = 100. * np.max(loss_std * ci_factor /  loss_avg)
	print('Max 95% CI relative to mean: ', max_deviate, '%')

	# fig, ax = plt.subplots(figsize=(7, 7))
	with sns.axes_style("white"):

		yticklabels = np.around(l2_grid * 1e5, decimals=3)
		yticklabels = np.round(yticklabels).astype(int)
		xticklabels = np.around(gam_grid, decimals=3)
		ax = sns.heatmap(loss_avg,  cmap="YlGnBu", xticklabels=xticklabels, yticklabels = yticklabels,  annot=True, annot_kws={"size": 8})
		ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
		plt.xlabel(r'Guidance Discount Factor $\gamma$')
		plt.ylabel(r'$L_2$ Regularization Factor [1e-5]')
		if save_PDF:
			save_fig(args.run_name)
		else:
			plt.title('Loss avg. Max 95% CI relative to mean: {}%\n {}'.format(np.around(max_deviate, decimals=1), args.run_name))
		plt.show()
		print('done')

In [ ]:
# ===== File: plot_reg_coeff.py =====
import numpy as np
import matplotlib.pyplot as plt

set_default_plot_params

plt.figure()

gammaDivGamma_e = np.linspace(0.1, 0.99, num=1000)
# gammaEval = 0.99
# S = 10
# A = 2
# n_params = S * A
# coeff = (gammaEval - gamma ) / (2 * gamma)
coeff = 0.5 * (1 / gammaDivGamma_e - 1)
# coeff /= n_params
plt.plot(gammaDivGamma_e, (coeff))


plt.grid(True)
plt.xlabel(r'$\gamma / \gamma_e$')
plt.ylabel(r'$\lambda = \frac{\gamma_e - \gamma}{2\gamma}$')
# plt.legend()
#
save_PDF = True  # False \ True
if save_PDF:
	save_fig('reg_coeff')
else:
	plt.title('Activation Reg. Coeff.')

plt.show()

print('done')

## 3) TD3/DDPG (MuJoCo)

Questa parte richiede tipicamente **PyTorch**, **gym** e un’installazione funzionante di **MuJoCo**.


In [ ]:
# ===== File: TD3_Code/utils.py =====
import numpy as np
import torch

def add_weight_decay(net, l2_value, skip_list=()):
	decay, no_decay = [], []
	for name, param in net.named_parameters():
		if not param.requires_grad:
			continue # frozen weights
		if name.endswith(".bias") or name in skip_list:
			no_decay.append(param)
		else:
			decay.append(param)
	return [{'params': no_decay, 'weight_decay': 0.}, {'params': decay, 'weight_decay': l2_value}]


class ReplayBuffer(object):
	def __init__(self, state_dim, action_dim, max_size=int(1e6)):
		self.max_size = max_size
		self.ptr = 0
		self.size = 0

		self.state = np.zeros((max_size, state_dim))
		self.action = np.zeros((max_size, action_dim))
		self.next_state = np.zeros((max_size, state_dim))
		self.reward = np.zeros((max_size, 1))
		self.not_done = np.zeros((max_size, 1))

		self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


	def add(self, state, action, next_state, reward, done):
		self.state[self.ptr] = state
		self.action[self.ptr] = action
		self.next_state[self.ptr] = next_state
		self.reward[self.ptr] = reward
		self.not_done[self.ptr] = 1. - done

		self.ptr = (self.ptr + 1) % self.max_size
		self.size = min(self.size + 1, self.max_size)


	def sample(self, batch_size):
		ind = np.random.randint(0, self.size, size=batch_size)

		return (
			torch.FloatTensor(self.state[ind]).to(self.device),
			torch.FloatTensor(self.action[ind]).to(self.device),
			torch.FloatTensor(self.next_state[ind]).to(self.device),
			torch.FloatTensor(self.reward[ind]).to(self.device),
			torch.FloatTensor(self.not_done[ind]).to(self.device)
		)

In [ ]:
# ===== File: TD3_Code/DDPG.py =====
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Implementation of Deep Deterministic Policy Gradients (DDPG)
# Paper: https://arxiv.org/abs/1509.02971
# [Not the implementation used in the TD3 paper]


class Actor(nn.Module):
	def __init__(self, state_dim, action_dim, max_action):
		super(Actor, self).__init__()

		self.l1 = nn.Linear(state_dim, 400)
		self.l2 = nn.Linear(400, 300)
		self.l3 = nn.Linear(300, action_dim)
		
		self.max_action = max_action

	
	def forward(self, state):
		a = F.relu(self.l1(state))
		a = F.relu(self.l2(a))
		return self.max_action * torch.tanh(self.l3(a))


class Critic(nn.Module):
	def __init__(self, state_dim, action_dim):
		super(Critic, self).__init__()

		self.l1 = nn.Linear(state_dim, 400)
		self.l2 = nn.Linear(400 + action_dim, 300)
		self.l3 = nn.Linear(300, 1)


	def forward(self, state, action):
		q = F.relu(self.l1(state))
		q = F.relu(self.l2(torch.cat([q, action], 1)))
		return self.l3(q)


class DDPG(object):
	def __init__(self, state_dim, action_dim, max_action, discount=0.99, tau=0.001):
		self.actor = Actor(state_dim, action_dim, max_action).to(device)
		self.actor_target = copy.deepcopy(self.actor)
		self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=1e-4)

		self.critic = Critic(state_dim, action_dim).to(device)
		self.critic_target = copy.deepcopy(self.critic)
		self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), weight_decay=1e-2)

		self.discount = discount
		self.tau = tau


	def select_action(self, state):
		state = torch.FloatTensor(state.reshape(1, -1)).to(device)
		return self.actor(state).cpu().data.numpy().flatten()


	def train(self, replay_buffer, batch_size=64):
		# Sample replay buffer 
		state, action, next_state, reward, not_done = replay_buffer.sample(batch_size)

		# Compute the target Q value
		target_Q = self.critic_target(next_state, self.actor_target(next_state))
		target_Q = reward + (not_done * self.discount * target_Q).detach()

		# Get current Q estimate
		current_Q = self.critic(state, action)

		# Compute critic loss
		critic_loss = F.mse_loss(current_Q, target_Q)

		# Optimize the critic
		self.critic_optimizer.zero_grad()
		critic_loss.backward()
		self.critic_optimizer.step()

		# Compute actor loss
		actor_loss = -self.critic(state, self.actor(state)).mean()
		
		# Optimize the actor 
		self.actor_optimizer.zero_grad()
		actor_loss.backward()
		self.actor_optimizer.step()

		# Update the frozen target models
		for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
			target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

		for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):
			target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)


	def save(self, filename):
		torch.save(self.critic.state_dict(), filename + "_critic")
		torch.save(self.critic_optimizer.state_dict(), filename + "_critic_optimizer")
		torch.save(self.actor.state_dict(), filename + "_actor")
		torch.save(self.actor_optimizer.state_dict(), filename + "_actor_optimizer")


	def load(self, filename):
		self.critic.load_state_dict(torch.load(filename + "_critic"))
		self.critic_optimizer.load_state_dict(torch.load(filename + "_critic_optimizer"))
		self.actor.load_state_dict(torch.load(filename + "_actor"))
		self.actor_optimizer.load_state_dict(torch.load(filename + "_actor_optimizer"))



In [ ]:
# ===== File: TD3_Code/OurDDPG.py =====
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from TD3_Code.utils import add_weight_decay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Re-tuned version of Deep Deterministic Policy Gradients (DDPG)
# Paper: https://arxiv.org/abs/1509.02971


class Actor(nn.Module):
	def __init__(self, state_dim, action_dim, max_action, actor_hiddens):
		super(Actor, self).__init__()

		self.l1 = nn.Linear(state_dim, actor_hiddens[0])
		self.l2 = nn.Linear(actor_hiddens[0], actor_hiddens[1])
		self.l3 = nn.Linear(actor_hiddens[1], action_dim)
		
		self.max_action = max_action

	
	def forward(self, state):
		a = F.relu(self.l1(state))
		a = F.relu(self.l2(a))
		return self.max_action * torch.tanh(self.l3(a))


class Critic(nn.Module):
	def __init__(self, state_dim, action_dim, critic_hiddens):
		super(Critic, self).__init__()

		self.l1 = nn.Linear(state_dim + action_dim, critic_hiddens[0])
		self.l2 = nn.Linear(critic_hiddens[0], critic_hiddens[1])
		self.l3 = nn.Linear(critic_hiddens[1], 1)


	def forward(self, state, action):
		q = F.relu(self.l1(torch.cat([state, action], 1)))
		q = F.relu(self.l2(q))
		return self.l3(q)



class DDPG(object):
	def __init__(self, state_dim, action_dim, max_action, actor_hiddens, critic_hiddens, discount=0.99, tau=0.005, actor_l2_reg=0, critic_l2_reg=0):
		self.actor = Actor(state_dim, action_dim, max_action, actor_hiddens).to(device)
		self.actor_target = copy.deepcopy(self.actor)


		actor_weights = add_weight_decay(self.actor, l2_value=actor_l2_reg)
		self.actor_optimizer = torch.optim.Adam(actor_weights, lr=3e-4)


		self.critic = Critic(state_dim, action_dim, critic_hiddens).to(device)
		self.critic_target = copy.deepcopy(self.critic)
		critic_weights = add_weight_decay(self.critic, l2_value=critic_l2_reg)
		self.critic_optimizer = torch.optim.Adam(critic_weights, lr=3e-4)

		self.discount = discount
		self.tau = tau


	def select_action(self, state):
		state = torch.FloatTensor(state.reshape(1, -1)).to(device)
		return self.actor(state).cpu().data.numpy().flatten()


	def train(self, replay_buffer, batch_size=100):
		# Sample replay buffer 
		state, action, next_state, reward, not_done = replay_buffer.sample(batch_size)

		# Compute the target Q value
		target_Q = self.critic_target(next_state, self.actor_target(next_state))
		target_Q = reward + (not_done * self.discount * target_Q).detach()

		# Get current Q estimate
		current_Q = self.critic(state, action)

		# Compute critic loss
		critic_loss = F.mse_loss(current_Q, target_Q)

		# Optimize the critic
		self.critic_optimizer.zero_grad()
		critic_loss.backward()
		self.critic_optimizer.step()

		# Compute actor loss
		actor_loss = -self.critic(state, self.actor(state)).mean()
		
		# Optimize the actor 
		self.actor_optimizer.zero_grad()
		actor_loss.backward()
		self.actor_optimizer.step()

		# Update the frozen target models
		for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
			target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

		for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):
			target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)


	def save(self, filename):
		torch.save(self.critic.state_dict(), filename + "_critic")
		torch.save(self.critic_optimizer.state_dict(), filename + "_critic_optimizer")
		torch.save(self.actor.state_dict(), filename + "_actor")
		torch.save(self.actor_optimizer.state_dict(), filename + "_actor_optimizer")


	def load(self, filename):
		self.critic.load_state_dict(torch.load(filename + "_critic"))
		self.critic_optimizer.load_state_dict(torch.load(filename + "_critic_optimizer"))
		self.actor.load_state_dict(torch.load(filename + "_actor"))
		self.actor_optimizer.load_state_dict(torch.load(filename + "_actor_optimizer"))



In [ ]:
# ===== File: TD3_Code/TD3.py =====
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from TD3_Code.utils import add_weight_decay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Implementation of Twin Delayed Deep Deterministic Policy Gradients (TD3)
# Paper: https://arxiv.org/abs/1802.09477


class Actor(nn.Module):
	def __init__(self, state_dim, action_dim, max_action, actor_hiddens):
		super(Actor, self).__init__()

		self.l1 = nn.Linear(state_dim, actor_hiddens[0])
		self.l2 = nn.Linear(actor_hiddens[0], actor_hiddens[1])
		self.l3 = nn.Linear(actor_hiddens[1], action_dim)
		
		self.max_action = max_action
		

	def forward(self, state):
		a = F.relu(self.l1(state))
		a = F.relu(self.l2(a))
		return self.max_action * torch.tanh(self.l3(a))


class Critic(nn.Module):
	def __init__(self, state_dim, action_dim, critic_hiddens):
		super(Critic, self).__init__()

		# Q1 architecture
		self.l1 = nn.Linear(state_dim + action_dim, critic_hiddens[0])
		self.l2 = nn.Linear(critic_hiddens[0], critic_hiddens[1])
		self.l3 = nn.Linear(critic_hiddens[1], 1)

		# Q2 architecture
		self.l4 = nn.Linear(state_dim + action_dim,  critic_hiddens[0])
		self.l5  = nn.Linear(critic_hiddens[0], critic_hiddens[1])
		self.l6 = nn.Linear(critic_hiddens[1], 1)


	def forward(self, state, action):
		sa = torch.cat([state, action], 1)

		q1 = F.relu(self.l1(sa))
		q1 = F.relu(self.l2(q1))
		q1 = self.l3(q1)

		q2 = F.relu(self.l4(sa))
		q2 = F.relu(self.l5(q2))
		q2 = self.l6(q2)
		return q1, q2


	def Q1(self, state, action):
		sa = torch.cat([state, action], 1)

		q1 = F.relu(self.l1(sa))
		q1 = F.relu(self.l2(q1))
		q1 = self.l3(q1)
		return q1


class TD3(object):
	def __init__(
		self,
		state_dim,
		action_dim,
		actor_hiddens,
		critic_hiddens,
		actor_l2_reg,
		critic_l2_reg,
		max_action,
		discount=0.99,
		tau=0.005,
		policy_noise=0.2,
		noise_clip=0.5,
		policy_freq=2
	):

		self.actor = Actor(state_dim, action_dim, max_action, actor_hiddens).to(device)
		self.actor_target = copy.deepcopy(self.actor)
		#  add weight decay parameter to weights not to biases
		actor_weights = add_weight_decay(self.actor, l2_value=actor_l2_reg)
		self.actor_optimizer = torch.optim.Adam(actor_weights, lr=3e-4)

		self.critic = Critic(state_dim, action_dim, critic_hiddens).to(device)
		self.critic_target = copy.deepcopy(self.critic)
		# add weight decay parameter to weights not to biases
		critic_weights = add_weight_decay(self.critic, l2_value=critic_l2_reg)
		self.critic_optimizer = torch.optim.Adam(critic_weights, lr=3e-4)

		self.max_action = max_action
		self.discount = discount
		self.tau = tau
		self.policy_noise = policy_noise
		self.noise_clip = noise_clip
		self.policy_freq = policy_freq
		self.actor_hiddens = actor_hiddens
		self.critic_hiddens = critic_hiddens

		self.total_it = 0


	def select_action(self, state):
		state = torch.FloatTensor(state.reshape(1, -1)).to(device)
		return self.actor(state).cpu().data.numpy().flatten()


	def train(self, replay_buffer, batch_size=100):
		self.total_it += 1

		# Sample replay buffer 
		state, action, next_state, reward, not_done = replay_buffer.sample(batch_size)

		with torch.no_grad():
			# Select action according to policy and add clipped noise
			noise = (
				torch.randn_like(action) * self.policy_noise
			).clamp(-self.noise_clip, self.noise_clip)
			
			next_action = (
				self.actor_target(next_state) + noise
			).clamp(-self.max_action, self.max_action)

			# Compute the target Q value
			target_Q1, target_Q2 = self.critic_target(next_state, next_action)
			target_Q = torch.min(target_Q1, target_Q2)
			target_Q = reward + not_done * self.discount * target_Q

		# Get current Q estimates
		current_Q1, current_Q2 = self.critic(state, action)

		# Compute critic loss
		critic_loss = F.mse_loss(current_Q1, target_Q) + F.mse_loss(current_Q2, target_Q)

		# Optimize the critic
		self.critic_optimizer.zero_grad()
		critic_loss.backward()
		self.critic_optimizer.step()

		# Delayed policy updates
		if self.total_it % self.policy_freq == 0:

			# Compute actor losse
			actor_loss = -self.critic.Q1(state, self.actor(state)).mean()
			
			# Optimize the actor 
			self.actor_optimizer.zero_grad()
			actor_loss.backward()
			self.actor_optimizer.step()

			# Update the frozen target models
			for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
				target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

			for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):
				target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)


	def save(self, filename):
		torch.save(self.critic.state_dict(), filename + "_critic")
		torch.save(self.critic_optimizer.state_dict(), filename + "_critic_optimizer")
		torch.save(self.actor.state_dict(), filename + "_actor")
		torch.save(self.actor_optimizer.state_dict(), filename + "_actor_optimizer")


	def load(self, filename):
		self.critic.load_state_dict(torch.load(filename + "_critic"))
		self.critic_optimizer.load_state_dict(torch.load(filename + "_critic_optimizer"))
		self.actor.load_state_dict(torch.load(filename + "_actor"))
		self.actor_optimizer.load_state_dict(torch.load(filename + "_actor_optimizer"))


In [ ]:
# ===== File: TD3_Code/mainTD3.py =====
import numpy as np
import torch
import gym
import argparse
import os
import pickle

from TD3_Code import utils, TD3, OurDDPG, DDPG
import json
# import warnings

# ---------------------------------------------------------------------------------------------------------------------------------#

# Runs policy for X episodes and returns average reward
# A fixed seed is used for the eval environment
def eval_policy(policy, env_name, seed, eval_episodes=10):
    eval_env = gym.make(env_name)
    eval_env.seed(seed + 100)

    avg_reward = 0.
    for _ in range(eval_episodes):
        state, done = eval_env.reset(), False
        while not done:
            action = policy.select_action(np.array(state))
            state, reward, done, _ = eval_env.step(action)
            avg_reward += reward

    avg_reward /= eval_episodes

    print("---------------------------------------")
    print(f"Evaluation over {eval_episodes} episodes: {avg_reward:.3f}")
    print("---------------------------------------")
    return avg_reward


# ---------------------------------------------------------------------------------------------------------------------------------#

def run_simulation_TD3(args, job_info=None):
    # warnings.filterwarnings("ignore", message="Box bound precision lowered by casting to float32")
    # warnings.filterwarnings("ignore", message="WARN: gym.spaces.Box autodetected dtype")

    if "alg" not in args:
        args.alg = args.policy

    file_name = f"{args.alg}_{args.env}_{args.seed}_{args.run_name}"
    print("---------------------------------------")
    print(f"Policy: {args.alg}, Env: {args.env}, Seed: {args.seed}, SimName: {args.run_name}")
    print("---------------------------------------")

    if not os.path.exists("./results"):
        os.makedirs("./results")

    if args.save_model and not os.path.exists("./models"):
        os.makedirs("./models")

    env = gym.make(args.env)

    # Set seeds
    env.seed(args.seed)
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    max_action = float(env.action_space.high[0])

    kwargs = {
        "state_dim": state_dim,
        "action_dim": action_dim,
        "actor_hiddens": args.actor_hiddens,
        "critic_hiddens": args.critic_hiddens,
        "actor_l2_reg": args.actor_l2_reg,
        "critic_l2_reg": args.critic_l2_reg,
        "max_action": max_action,
        "discount": args.discount,
        "tau": args.tau,
    }

    # Initialize policy
    if args.alg == "TD3":
        # Target policy smoothing is scaled wrt the action scale
        kwargs["policy_noise"] = args.policy_noise * max_action
        kwargs["noise_clip"] = args.noise_clip * max_action
        kwargs["policy_freq"] = args.policy_freq
        policy = TD3.TD3(**kwargs)
    elif args.alg == "OurDDPG":
        policy = OurDDPG.DDPG(**kwargs)
    elif args.alg == "DDPG":
        policy = DDPG.DDPG(**kwargs)

    if args.load_model != "":
        policy_file = file_name if args.load_model == "default" else args.load_model
        policy.load(f"./models/{policy_file}")

    replay_buffer = utils.ReplayBuffer(state_dim, action_dim)

    # Evaluate untrained policy
    evaluations = []
    timesteps_snapshots = []

    state, done = env.reset(), False
    episode_reward = 0
    episode_timesteps = 0
    episode_num = 0
    eval_reward = None

    for t in range(int(args.max_timesteps)):

        episode_timesteps += 1

        # Select action randomly or according to policy
        if t < args.start_timesteps:
            action = env.action_space.sample()
        else:
            action = (
                    policy.select_action(np.array(state))
                    + np.random.normal(0, max_action * args.expl_noise, size=action_dim)
            ).clip(-max_action, max_action)

        # Perform action
        next_state, reward, done, _ = env.step(action)
        done_bool = float(done) if episode_timesteps < env._max_episode_steps else 0

        # Store data in replay buffer
        replay_buffer.add(state, action, next_state, reward, done_bool)

        state = next_state
        episode_reward += reward

        # Train agent after collecting sufficient data
        if t >= args.start_timesteps:
            for i_iter in range(args.iter_per_sample):
                policy.train(replay_buffer, args.batch_size)
            # end for i_iter
        # end if

        if done:
            # +1 to account for 0 indexing. +0 on ep_timesteps since it will increment +1 even if done=True
            print(
                f"Total T: {t + 1} Episode Num: {episode_num + 1} Episode T: {episode_timesteps} Reward: {episode_reward:.3f}")
            # Reset environment
            state, done = env.reset(), False
            episode_reward = 0
            episode_timesteps = 0
            episode_num += 1
        # end if

        # Evaluate
        if (t + 1) % args.eval_freq == 0:
            eval_reward = eval_policy(policy, args.env, args.seed, args.evaluation_num_episodes)
            evaluations.append(eval_reward)
            np.save(f"./results/{file_name}", evaluations)
            timesteps_snapshots.append(t+1)
            if args.save_model:
                policy.save(f"./models/{file_name}")
            # end if
            f_path = os.path.join(args.result_dir, 'jobs', args.job_name) + ".p"
            save_dict = {'job_info': job_info,
                         'timesteps_snapshots': timesteps_snapshots,
                         'evaluations': evaluations}
            pickle.dump(save_dict, open(f_path, "wb"))
        # end if
    #  end for t
    print("final_eval_reward: ", eval_reward)
    return eval_reward
# end run_simulation

# ---------------------------------------------------------------------------------------------------------------------------------#


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--alg", default="TD3")  # Policy name (TD3, DDPG or OurDDPG)
    parser.add_argument("--env", default="HalfCheetah-v2")  # OpenAI gym environment name
    parser.add_argument("--seed", default=0, type=int)  # Sets Gym, PyTorch and Numpy seeds
    parser.add_argument("--start_timesteps", default=1e4, type=int)  # Time steps initial random policy is used
    parser.add_argument("--eval_freq", default=5e3, type=int)  # How often (time steps) we evaluate
    parser.add_argument("--max_timesteps", default=1e6, type=int)  # Max time steps to run environment
    parser.add_argument("--expl_noise", default=0.1)  # Std of Gaussian exploration noise
    parser.add_argument("--batch_size", default=256, type=int)  # Batch size for both actor and critic
    parser.add_argument("--discount", default=0.99)  # Discount factor
    parser.add_argument("--tau", default=0.005)  # Target network update rate
    parser.add_argument("--policy_noise", default=0.2)  # Noise added to target policy during critic update
    parser.add_argument("--noise_clip", default=0.5)  # Range to clip target policy noise
    parser.add_argument("--policy_freq", default=2, type=int)  # Frequency of delayed policy updates
    parser.add_argument("--save_model", action="store_true")  # Save model and optimizer parameters
    parser.add_argument("--load_model", default="")  # Model load file name, "" doesn't load, "default" uses file_name
    parser.add_argument("--run_name", default="")  #
    parser.add_argument("--iter_per_sample", default=1)  #
    parser.add_argument("--actor_hiddens", default='[400, 300]', type=json.loads)
    parser.add_argument("--critic_hiddens", default='[400, 300]', type=json.loads)
    parser.add_argument("--critic_l2_reg", default=0)  #  L2 regularization factor for the Q-networks (critic)
    parser.add_argument("--actor_l2_reg", default=0)  # L2 regularization factor for the policy-networks (actor)

    args = parser.parse_args()
    run_simulation_TD3(args)


## 4) Suggerimenti pratici

- Se vuoi “ripartire pulito” tra un esperimento e l’altro, riavvia il kernel.
- Le directory di output vengono create dagli script (spesso sotto `./saved/...`). Se preferisci usare `NOTEBOOK_OUT`, puoi cambiare `create_result_dir(...)` o i percorsi `result_dir_to_load` nelle celle.

---
